In [1]:
##############################################################################################
#Prerequisites for summarising GIPT data in tables and updating Google sheets
##############################################################################################

!pip install pygsheets
#!pip install gspread_pandas #another package for intereacting with Google sheets but not currently using - examples using gspread here: https://stackoverflow.com/questions/48470691/access-google-sheets-on-google-colaboratory
#drive.flush_and_unmount()
from google.colab import auth
auth.authenticate_user()
from google.auth import default
creds, _ = default()

from google.colab import drive
drive.mount('/content/drive')

import pandas
import pandas as pd
import numpy as np
import pygsheets
import re
#import gspread_pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 kB 2.8 MB/s eta 0:00:00
Mounted at /content/drive


In [13]:
###############################################
#LOAD GIPT datasheet
###############################################
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power March 2024.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power April 2024.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power May 2024.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power June 2024.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power July 2024.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power August 2024.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power September 2024.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power January 2025.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power February 2025.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power February 2025 update II.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power March 2025.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power April 2025.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power July 2025.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power August 2025.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power September 2025.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Data compilations/Global Integrated Power September 2025 II.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/GIPT public release files/Global Integrated Power January 2026.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/GIPT public release files/Global Integrated Power February 2026.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/GIPT public release files/Global Integrated Power February 2026 II.xlsx"
#gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/GIPT public release files/Global Integrated Power March 2026.xlsx"
gipt_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/GIPT public release files/Global Integrated Power March 2026 II.xlsx"
gipt=pandas.read_excel(gipt_file, sheet_name='Power facilities')

###############################################
# CUSTOM REGIONS DEFINITIONS FROM GIPT DATASHEET
###############################################
defs=pandas.read_excel(gipt_file, sheet_name='Regions, area, and countries')

###############################################
# MAKE SOME CUSTOM ADJUSTMENTS
###############################################
# Adjust 'not found' values in GIPT
gipt.loc[gipt['Capacity (MW)']=='not found','Capacity (MW)']=np.nan
# Change 'Capacity' to floats
gipt['Capacity (MW)']=gipt['Capacity (MW)'].astype(float)
gipt['Capacity (MW)']=gipt['Capacity (MW)'].fillna(0.0)

#Power sector summary tables differ in treatment of subthreshold data (see here for conventions: https://docs.google.com/document/d/1QA7c8eqhISiTO5r0fI4fOhTN-FKQoOhWfZGwgEfbUoY/edit?tab=t.0)

#Take out wind under 10MW
gipt.drop(gipt[(gipt.Type=='wind')&(gipt['Capacity (MW)']<10)].index, inplace=True)

#Take out bioenergy under 10WM ### threshold removed Sept 2025
#gipt.drop(gipt[(gipt.Type=='bioenergy')&(gipt['Capacity (MW)']<10)].index, inplace=True)

#Take out oil/gas under threshold ### dropped March 2026
# eu_uk=["United Kingdom", "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic", "Denmark", "Estonia", "Finland", "France", "Germany", "Greece", "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta", "Netherlands", "Poland", "Portugal", "Romania", "Slovakia", "Slovenia", "Spain", "Sweden"]
# gipt.drop(gipt[(gipt.Type=='oil/gas')&(gipt['Capacity (MW)']<50)&(~gipt['Country/area'].isin(eu_uk))].index, inplace=True)
# gipt.drop(gipt[(gipt.Type=='oil/gas')&(gipt['Capacity (MW)']<20)&(gipt['Country/area'].isin(eu_uk))].index, inplace=True)

#Change status: cancelled - inferred 4 y / shelved - inferred 2 y
gipt.loc[gipt.Status=='cancelled - inferred 4 y','Status']='cancelled'
gipt.loc[gipt.Status=='shelved - inferred 2 y','Status']='shelved'



In [ ]:
# @title
##############################################################################################
#The Solar tracker harmonizes all capacity to MWac in summary tables
#See source code for doing this here: https://github.com/GlobalEnergyMonitor/Renewables_Others/blob/main/SolarCode/ConvertToMWac.py
#This portion of code needs to be run even for non-solar updates, otherwise a mix of DC and AC will get added to existing summary tables
#Note that this code is slow to run of Google Colab
##############################################################################################

#Load latest GSPT. This is needed as the GIPT doesn't include teh DC/AC data filed. Note, this source file requires updating when GSPT updates
df_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Renewables & Other Power Program/Shared Wind and Solar Folder/Previous Updates/Wind and Solar Update v4b H2 2025/(Feb 2026) Launch Documents/Download Files/Global-Solar-Power-Tracker-February-2026.xlsx"
df=pandas.read_excel(df_file, 'Utility-Scale (1 MW+)')

# this is the conversion between DC to AC. Value from TransitionZero
conversionFactor = 0.87

# this is the minimum count number in order to use country or subregion value rather than subregion or region value
minval = 30

# Incase you haven't already, remove everything below 1 MW. We cannot do this after we do the conversion since many projects will end up with less than 1 MW capacity
df = df[df['Capacity (MW)'] >= 1].reset_index(drop=True)

# save original capacity to a new column
df['Capacity (MW) orig'] = df['Capacity (MW)']

# if capacity rating is DC, convert to AC
df.loc[df['Capacity Rating'] == 'MWp/dc', 'Capacity (MW)'] = df['Capacity (MW) orig']*conversionFactor

# if capacity rating is unknown convert the value based on the probability it's MWac based on the country/subregion/region

# I think we don't want to have government datasets biasing this, so we won't include projects that have an other location or phase ID that's not WEPP or WikiSolar
## replace nans with blanks
df.fillna("", inplace=True)
## loop through every Other IDs location and Other IDs phase and remove WEPP & WKSL so that we can ignore anything with an entry in the Other IDs columns
for index, row in df.iterrows():
    loc_id = row['Other IDs (location)']
    phase_id = row['Other IDs (unit/phase)']
    # split the ID by commas
    loc_id_list = loc_id.split(",")
    phase_id_list = phase_id.split(",")
    # create a temporary list
    loc_tmp_lst = []
    phase_tmp_lst = []
    # remove any location ID that start with WEPP or WKSL
    for id in loc_id_list:
        id = id.strip()
        if id.startswith("WEPP") | id.startswith("WKSL"):
            pass
        else:
            loc_tmp_lst.append(id)

    # remove any location ID that start with WEPP
    for id in phase_id_list:
        id = id.strip()
        if id.startswith("WEPP") | id.startswith("WKSL"):
            pass
        else:
            phase_tmp_lst.append(id)
    # join tmp_lst together as a comma delimited string
    new_loc_id = ",".join(map(str, loc_tmp_lst))
    new_phase_id = ",".join(map(str, phase_tmp_lst))
    # write this to the dataframe row
    df.loc[index, 'Other IDs (location)'] = new_loc_id
    df.loc[index, 'Other IDs (unit/phase)'] = new_phase_id


## compute liklihood based on the region. If there are no ac or dc counts in the region set the region probability to 50%
regions = df['Region'].unique().tolist()
region_prob = []
for region in regions:
    countsac = len(df[(df['Region'] == region) & (df['Capacity Rating'] == 'MWac') & (df['Other IDs (unit/phase)'] == '') & (df['Other IDs (location)'] == '')])
    countsdc = len(df[(df['Region'] == region) & (df['Capacity Rating'] == 'MWp/dc') & (df['Other IDs (unit/phase)'] == '') & (df['Other IDs (location)'] == '')])
    if (countsac+countsdc) > 0:
        region_prob.append(countsac/(countsac+countsdc))
    else:
        region_prob.append(0.5)

## compute liklihood based on the sub-region. If ac+dc counts are less than minval use region numbers
subregions = df['Subregion'].unique().tolist()
subregion_prob = []
for subregion in subregions:
    countsac = len(df[(df['Subregion'] == subregion) & (df['Capacity Rating'] == 'MWac') & (
                df['Other IDs (unit/phase)'] == '') & (df['Other IDs (location)'] == '')])
    countsdc = len(df[(df['Subregion'] == subregion) & (df['Capacity Rating'] == 'MWp/dc') & (
                df['Other IDs (unit/phase)'] == '') & (df['Other IDs (location)'] == '')])
    if countsac+countsdc >= minval:
        subregion_prob.append(countsac / (countsac + countsdc))
    else:
        # get the region associated with subregions
        rgn = df.loc[df['Subregion'] == subregion, 'Region'].iloc[0]
        # find that region's probability
        idx = regions.index(rgn)
        subregion_prob.append(region_prob[idx])

## compute liklihood based on the country. If ac+dc counts are less than 50 use subregion numbers
countries = df['Country/Area'].unique().tolist()
country_prob = []
for country in countries:
    countsac = len(df[(df['Country/Area'] == country) & (df['Capacity Rating'] == 'MWac') & (
                df['Other IDs (unit/phase)'] == '') & (df['Other IDs (location)'] == '')])
    countsdc = len(df[(df['Country/Area'] == country) & (df['Capacity Rating'] == 'MWp/dc') & (
                df['Other IDs (unit/phase)'] == '') & (df['Other IDs (location)'] == '')])
    if countsac+countsdc >= minval:
        country_prob.append(countsac / (countsac + countsdc))
    else:
        # get the subregion associated with country
        subr = df.loc[df['Country/Area'] == country, 'Subregion'].iloc[0]
        # find that region's probability
        idx = subregions.index(subr)
        country_prob.append(subregion_prob[idx])

# Adjust 'unknown' capacities to MWac
for index, row in df.iterrows():
    if row['Capacity Rating'] == 'unknown':
        idx = countries.index(row['Country/Area'])
        try:
            df.at[index, 'Capacity (MW)'] = ((1-conversionFactor)*country_prob[idx] + conversionFactor)*row['Capacity (MW) orig']
        except:
            print(row)

# save original capacity rating to a new column and set capacity rating to MWac
df['Capacity Rating orig'] = df['Capacity Rating']
df['Capacity Rating'] = 'MWac'



#Replace gipt capacity values for solar with AC converted values
gipt=gipt.set_index('GEM unit/phase ID')
gipt.loc[df['GEM phase ID'],'Capacity (MW)']=df['Capacity (MW)'].values


/tmp/ipykernel_14934/525396341.py:31: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("", inplace=True)


In [14]:
##############################################################################################
#The Solar tracker harmonizes all capacity to MWac in summary tables
#See source code for doing this here: https://github.com/GlobalEnergyMonitor/Renewables_Others/blob/main/SolarCode/ConvertToMWac.py
#This has been refactored here to speed up the running the code in Collab
#This portion of code needs to be run even for non-solar updates, otherwise a mix of DC and AC will get added to existing summary tables
##############################################################################################

# -----------------------------
# load latest GSPT
# -----------------------------
df_file = "/content/drive/Shareddrives/GEM Shared Drive/Programs/Renewables & Other Power Program/Shared Wind and Solar Folder/Previous Updates/Wind and Solar Update v4b H2 2025/(Feb 2026) Launch Documents/Download Files/Global-Solar-Power-Tracker-February-2026.xlsx"
df = pd.read_excel(df_file, sheet_name="Utility-Scale (1 MW+)")

conversionFactor = 0.87
minval = 30

# -----------------------------
# basic cleaning / setup
# -----------------------------
df = df[df["Capacity (MW)"] >= 1].copy()
df["Capacity (MW) orig"] = df["Capacity (MW)"]

# fill only columns that need it, not whole df
id_cols = ["Other IDs (location)", "Other IDs (unit/phase)"]
for col in id_cols:
    df[col] = df[col].fillna("")

# convert known DC -> AC
mask_dc = df["Capacity Rating"].eq("MWp/dc")
df.loc[mask_dc, "Capacity (MW)"] = df.loc[mask_dc, "Capacity (MW) orig"] * conversionFactor

# -----------------------------
# strip WEPP / WKSL ids
# -----------------------------
def remove_wepp_wksl(s):
    if not s:
        return ""
    parts = [x.strip() for x in s.split(",")]
    parts = [x for x in parts if x and not (x.startswith("WEPP") or x.startswith("WKSL"))]
    return ",".join(parts)

df["Other IDs (location)"] = df["Other IDs (location)"].apply(remove_wepp_wksl)
df["Other IDs (unit/phase)"] = df["Other IDs (unit/phase)"].apply(remove_wepp_wksl)

# -----------------------------
# probability base subset:
# only rows with no non-WEPP/WKSL other IDs
# and only AC/DC known rows
# -----------------------------
base = df[
    df["Other IDs (location)"].eq("") &
    df["Other IDs (unit/phase)"].eq("") &
    df["Capacity Rating"].isin(["MWac", "MWp/dc"])
].copy()

# count AC/DC once for each level
# -----------------------------
# region probabilities
region_counts = (
    base.groupby(["Region", "Capacity Rating"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["MWac", "MWp/dc"], fill_value=0)
)

region_total = region_counts["MWac"] + region_counts["MWp/dc"]
region_prob = (region_counts["MWac"] / region_total).fillna(0.5)

# -----------------------------
# subregion probabilities
subregion_counts = (
    base.groupby(["Subregion", "Capacity Rating"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["MWac", "MWp/dc"], fill_value=0)
)

subregion_total = subregion_counts["MWac"] + subregion_counts["MWp/dc"]
subregion_direct_prob = subregion_counts["MWac"] / subregion_total

# map each subregion to its region
subregion_to_region = (
    df[["Subregion", "Region"]]
    .drop_duplicates(subset=["Subregion"])
    .set_index("Subregion")["Region"]
)

subregion_prob = subregion_direct_prob.copy()
fallback_subregions = subregion_total < minval
subregion_prob.loc[fallback_subregions] = (
    subregion_to_region.loc[fallback_subregions.index[fallback_subregions]]
    .map(region_prob)
    .values
)
subregion_prob = subregion_prob.fillna(0.5)

# -----------------------------
# country probabilities
country_counts = (
    base.groupby(["Country/Area", "Capacity Rating"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["MWac", "MWp/dc"], fill_value=0)
)

country_total = country_counts["MWac"] + country_counts["MWp/dc"]
country_direct_prob = country_counts["MWac"] / country_total

# map each country to its subregion
country_to_subregion = (
    df[["Country/Area", "Subregion"]]
    .drop_duplicates(subset=["Country/Area"])
    .set_index("Country/Area")["Subregion"]
)

country_prob = country_direct_prob.copy()
fallback_countries = country_total < minval
country_prob.loc[fallback_countries] = (
    country_to_subregion.loc[fallback_countries.index[fallback_countries]]
    .map(subregion_prob)
    .values
)
country_prob = country_prob.fillna(0.5)

# -----------------------------
# adjust unknown capacities vectorized
# formula:
# ((1-conversionFactor) * country_prob + conversionFactor) * orig_capacity
# -----------------------------
df["Capacity Rating orig"] = df["Capacity Rating"]

unknown_mask = df["Capacity Rating"].eq("unknown")
prob_for_rows = df["Country/Area"].map(country_prob).fillna(0.5)

df.loc[unknown_mask, "Capacity (MW)"] = (
    ((1 - conversionFactor) * prob_for_rows[unknown_mask] + conversionFactor)
    * df.loc[unknown_mask, "Capacity (MW) orig"]
)

# set everything to MWac
df["Capacity Rating"] = "MWac"

# -----------------------------
# replace GIPT solar capacities
# -----------------------------
gipt = gipt.set_index("GEM unit/phase ID")
gipt.loc[df["GEM phase ID"], "Capacity (MW)"] = df["Capacity (MW)"].values

In [ ]:
# @title
####################################################################################################################
#
#FEB 2026 ADDITION — DISTRIBUTED SOLAR ADDED TO SUMMARY TABLES —
#
####################################################################################################################

df_file="/content/drive/Shareddrives/GEM Shared Drive/Programs/Renewables & Other Power Program/Shared Wind and Solar Folder/Previous Updates/Wind and Solar Update v4b H2 2025/(Feb 2026) Launch Documents/Download Files/Global-Solar-Power-Tracker-February-2026.xlsx"
df=pandas.read_excel(df_file, 'Distributed (<1 MW)')


df=df[['Country/Area', 'Capacity (MW)','Status', 'Subregion', 'Region']]
df.rename(columns={'Country/Area':'Country/area'}, inplace=True)
df['Type']='solar dist'

gipt=pandas.concat([gipt,df])




In [5]:
####################################################################################################################
# 1) A FUNCTION TO HANDLE AGGREGATION BY: REGION, TECH, AND STATUS
# This updates the sheet called: "Power capacity by region and technology, grouped by status" https://docs.google.com/spreadsheets/d/1yV7GYPO_2Sx2ZMrRIJ0q2iryuZjIb4oeWJTacsuRoi4/edit#gid=1156264836
####################################################################################################################

regs1=['Africa','Northern Africa','Sub-Saharan Africa','Americas','Latin America and the Caribbean','Northern America','Asia','Central Asia','Eastern Asia','South-eastern Asia','Southern Asia','Western Asia',
'Europe','Eastern Europe','Northern Europe','Southern Europe','Western Europe','Oceania','Australia and New Zealand','Melanesia','Micronesia','Polynesia']
types=['coal', 'oil/gas', 'utility-scale solar','solar dist', 'wind', 'nuclear','hydropower', 'bioenergy','geothermal']
status=['announced','pre-permit','permitted','pre-construction','construction','shelved','operating','mothballed','cancelled','retired','inactive']

def _ensure_dimensions(df_to_process, index_levels, column_levels, index_name=None, fill_value=0.0):
    """
    Ensures a DataFrame has a consistent set of index levels and column levels,
    filling missing entries with a specified fill_value and reordering.

    Args:
        df_to_process (pd.DataFrame): The DataFrame to process.
        index_levels (list): Ordered list of desired index entries.
        column_levels (list): Ordered list of desired column entries.
        index_name (str, optional): The name for the index. Defaults to None.
        fill_value (float, optional): Value to fill for missing entries. Defaults to 0.0.

    Returns:
        pd.DataFrame: Processed DataFrame with ensured dimensions.
    """
    # Ensure all required columns are present
    current_columns = set(df_to_process.columns)
    missing_columns = list(set(column_levels) - current_columns)
    for col in missing_columns:
        df_to_process[col] = fill_value

    # Ensure all required index entries are present
    current_index = set(df_to_process.index)
    missing_index_entries = list(set(index_levels) - current_index)
    for idx_entry in missing_index_entries:
        df_to_process.loc[idx_entry] = fill_value

    # Reorder index and columns
    df_to_process = df_to_process.loc[index_levels, column_levels].fillna(fill_value)
    if index_name:
        df_to_process.index.name = index_name
    return df_to_process


def cap_by_region_and_tech(stat, dataframe=None):
    """
    Calculates power capacity by region/subregion and technology for a given status.
    This function replicates the exact output logic of the original `cap_by_region_and_tech`
    function, with improved readability and structure.
    It specifically handles bi-national hydropower projects by splitting their capacity
    between relevant countries before aggregation, then combines with other technology types.

    Args:
        stat (str): The status to filter projects by (e.g., 'operating', 'construction').
        dataframe (pd.DataFrame, optional): The input GIPT DataFrame. If None,
                                           it uses the global `gipt` DataFrame.

    Returns:
        pd.DataFrame: A DataFrame with power capacity aggregated by region/subregion
                      and technology, indexed by the provided status.
    """
    if dataframe is None:
        df = gipt.copy()
    else:
        df = dataframe.copy()

    # Filter projects for the given status
    df_filtered_status = df[df.Status == stat]

    # --- Step 1: Prepare Hydropower capacity, specifically handling bi-national projects ---
    # Identify mononational and bi-national hydropower projects
    mononational_hydro_projects = df_filtered_status[df_filtered_status['Country/area 2 (hydropower only)'].isnull()]
    binational_hydro_projects = df_filtered_status[df_filtered_status['Country/area 2 (hydropower only)'].notnull()]

    # Extract relevant columns and rename for consistency for bi-national projects
    binational_hydro_part1 = binational_hydro_projects[[
        'Country/area 2 (hydropower only)', 'Country/area 2 Capacity (MW) (hydropower only)',
        'Type', 'Region', 'Subregion'
    ]].rename(columns={
        'Country/area 2 (hydropower only)': 'Country/area',
        'Country/area 2 Capacity (MW) (hydropower only)': 'Capacity (MW)'
    })

    binational_hydro_part2 = binational_hydro_projects[[
        'Country/area 1 (hydropower only)', 'Country/area 1 Capacity (MW) (hydropower only)',
        'Type', 'Region', 'Subregion'
    ]].rename(columns={
        'Country/area 1 (hydropower only)': 'Country/area',
        'Country/area 1 Capacity (MW) (hydropower only)': 'Capacity (MW)'
    })

    # Combine all hydropower capacities that account for bi-national splits
    # For mononational, use the 'Country/area 1 Capacity (MW) (hydropower only)' for summing hydropower.
    # This is based on the logic observed in the original function.
    all_hydro_data_for_aggregation = pd.concat([
        mononational_hydro_projects[['Region', 'Subregion', 'Type', 'Country/area 1 Capacity (MW) (hydropower only)']]
            .rename(columns={'Country/area 1 Capacity (MW) (hydropower only)': 'Capacity (MW)'}),
        binational_hydro_part1[['Region', 'Subregion', 'Type', 'Capacity (MW)']],
        binational_hydro_part2[['Region', 'Subregion', 'Type', 'Capacity (MW)']]
    ])
    # Filter only 'hydropower' type from this adjusted dataset
    all_hydro_data_for_aggregation = all_hydro_data_for_aggregation[all_hydro_data_for_aggregation['Type'] == 'hydropower']


    # Aggregate adjusted hydropower data by Region and Subregion
    hydro_agg_region = all_hydro_data_for_aggregation.groupby(['Region', 'Type'])['Capacity (MW)'].sum().unstack(fill_value=0.0)
    hydro_agg_subregion = all_hydro_data_for_aggregation.groupby(['Subregion', 'Type'])['Capacity (MW)'].sum().unstack(fill_value=0.0)

    # Combine region and subregion aggregations for hydropower
    combined_hydro_agg = pd.concat([hydro_agg_region, hydro_agg_subregion], axis=0)

    # Ensure all regions/subregions and 'hydropower' column are present and ordered
    tmp_hydro_df = _ensure_dimensions(combined_hydro_agg, regs1, ['hydropower'])

    # --- Step 2: Aggregate capacity for ALL technology types directly from gipt['Capacity (MW)'] ---
    # This includes hydropower as well, which will be reconciled in the next step.
    agg_all_types_region = df_filtered_status.groupby(['Region', 'Type'])['Capacity (MW)'].sum().unstack(fill_value=0.0)
    agg_all_types_subregion = df_filtered_status.groupby(['Subregion', 'Type'])['Capacity (MW)'].sum().unstack(fill_value=0.0)

    # Combine region and subregion aggregations for all types
    combined_all_types_agg = pd.concat([agg_all_types_region, agg_all_types_subregion], axis=0)

    # Ensure all regions/subregions and all 'types' (including 'solar dist') are present and ordered
    tmp_all_types_df = _ensure_dimensions(combined_all_types_agg, regs1, types)

    # --- Step 3: Combine adjusted hydropower and other technologies as in original logic ---
    # The original code takes 'hydropower' from `tmp_hydro_df` and other technologies from `tmp_all_types_df`.
    # The 'solar dist' column is temporarily included then dropped to match original behavior.
    columns_to_add_from_all_types = ['coal','oil/gas','utility-scale solar','solar dist','wind','nuclear','bioenergy','geothermal']

    final_combined_capacities = tmp_hydro_df.add(
        tmp_all_types_df[columns_to_add_from_all_types], fill_value=0.0
    )

    # Final selection of columns and set index name to match original output
    final_output_columns = ['coal','oil/gas','utility-scale solar','wind','nuclear','hydropower','bioenergy','geothermal']
    final_df_result = final_combined_capacities[final_output_columns]
    final_df_result.index.name = stat

    return final_df_result

#test
#cap_by_region_and_tech('operating')[:20]

In [ ]:
################################################################################################################################################################
# 1) RUN THE FUNCTION ABOVE PER TAB IN TARGET SUMMARY TABLE: https://docs.google.com/spreadsheets/d/1yV7GYPO_2Sx2ZMrRIJ0q2iryuZjIb4oeWJTacsuRoi4/edit#gid=798700965
###############################################################################################################################################################

#AUTHORIZE EDITS
gc = pygsheets.authorize(service_file='/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Work in Progress/GDRIVE_API_CREDENTIALS.json')
spreadsheet = gc.open_by_key("1yV7GYPO_2Sx2ZMrRIJ0q2iryuZjIb4oeWJTacsuRoi4")
#ANNOUNCED TAB
test = spreadsheet.worksheet('title', 'Announced')
test.set_dataframe(cap_by_region_and_tech('announced'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Pre-construction TAB
pre_construction=cap_by_region_and_tech('pre-construction')
#pre_construction['coal']=(cap_by_region_and_tech('pre-permit')+cap_by_region_and_tech('permitted'))['coal'].values
test = spreadsheet.worksheet('title', 'Pre-construction')
test.set_dataframe(pre_construction, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Construction TAB
test = spreadsheet.worksheet('title', 'Construction')
test.set_dataframe(cap_by_region_and_tech('construction'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Shelved TAB
test = spreadsheet.worksheet('title', 'Shelved')
test.set_dataframe(cap_by_region_and_tech('shelved'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Operating TAB
test = spreadsheet.worksheet('title', 'Operating')
test.set_dataframe(cap_by_region_and_tech('operating'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Mothballed TAB
test = spreadsheet.worksheet('title', 'Mothballed')
test.set_dataframe(cap_by_region_and_tech('mothballed'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Cancelled TAB
test = spreadsheet.worksheet('title', 'Cancelled')
test.set_dataframe(cap_by_region_and_tech('cancelled'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Retired TAB
test = spreadsheet.worksheet('title', 'Retired')
test.set_dataframe(cap_by_region_and_tech('retired'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)


In [ ]:
##########################################################################################################################################
# 2) A FUNCTION TO HANDLE AGGREGATION BY: COUNTRY, TECH, AND STATUS
# This updates the sheet called: "Power capacity by country and technology, grouped by status" https://docs.google.com/spreadsheets/d/1a1s9hHB9bw-WX_RgUDseTwq6ALqu3M_4P7iMkKHxlEA/edit#gid=42360804
##########################################################################################################################################

def cap_by_country_and_tech(stat, dataframe=None):
    """
    Calculates power capacity by country and technology for a given status,
    handling bi-national hydropower projects by splitting their capacity
    between relevant countries before aggregation.

    Args:
        stat (str): The status to filter projects by (e.g., 'operating', 'construction').
        dataframe (pd.DataFrame, optional): The input GIPT DataFrame. If None,
                                           it uses the global `gipt` DataFrame.

    Returns:
        pd.DataFrame: A DataFrame with power capacity aggregated by country
                      and technology, indexed by the provided status.
                      Includes 'region' and 'sub-region' columns.
    """
    if dataframe is None:
        df = gipt.copy()
    else:
        df = dataframe.copy()

    # Filter projects for the given status
    df_filtered_status = df[df.Status == stat].copy()

    # Get a comprehensive list of all countries for consistent indexing
    all_countries_in_gipt = np.sort(gipt['Country/area'].unique())

    # --- Step 1: Prepare Hydropower capacity, specifically handling bi-national projects ---
    mononational_hydro_projects = df_filtered_status[df_filtered_status['Country/area 2 (hydropower only)'].isnull()].copy()
    binational_hydro_projects = df_filtered_status[df_filtered_status['Country/area 2 (hydropower only)'].notnull()].copy()

    # Extract relevant columns and rename for consistency for bi-national projects
    binational_hydro_part1 = binational_hydro_projects[[
        'Country/area 2 (hydropower only)', 'Country/area 2 Capacity (MW) (hydropower only)', 'Type'
    ]].rename(columns={
        'Country/area 2 (hydropower only)': 'Country/area',
        'Country/area 2 Capacity (MW) (hydropower only)': 'Capacity (MW)'
    })

    binational_hydro_part2 = binational_hydro_projects[[
        'Country/area 1 (hydropower only)', 'Country/area 1 Capacity (MW) (hydropower only)', 'Type'
    ]].rename(columns={
        'Country/area 1 (hydropower only)': 'Country/area',
        'Country/area 1 Capacity (MW) (hydropower only)': 'Capacity (MW)'
    })

    # Combine all hydropower capacities that account for bi-national splits
    all_hydro_data_for_aggregation = pd.concat([
        mononational_hydro_projects[['Country/area', 'Type', 'Country/area 1 Capacity (MW) (hydropower only)']]
            .rename(columns={'Country/area 1 Capacity (MW) (hydropower only)': 'Capacity (MW)'}),
        binational_hydro_part1,
        binational_hydro_part2
    ])

    # Filter only 'hydropower' type from this adjusted dataset
    hydro_agg_df = all_hydro_data_for_aggregation[all_hydro_data_for_aggregation['Type'] == 'hydropower'] \
        .groupby(['Country/area', 'Type'])['Capacity (MW)'].sum().unstack(fill_value=0.0)

    # --- Step 2: Aggregate capacity for ALL technology types directly from gipt['Capacity (MW)'] ---
    agg_all_types_df = df_filtered_status.groupby(['Country/area', 'Type'])['Capacity (MW)'].sum().unstack(fill_value=0.0)

    # --- Step 3: Combine adjusted hydropower and other technologies ---
    # Start with the general aggregation, then replace or add hydropower from the adjusted hydro_agg_df
    final_df = agg_all_types_df.copy()

    # Update or add the 'hydropower' column from the adjusted aggregation
    if 'hydropower' in hydro_agg_df.columns:
        final_df['hydropower'] = hydro_agg_df['hydropower']
    else:
        final_df['hydropower'] = 0.0
    final_df['hydropower'] = final_df['hydropower'].fillna(0.0) # Ensure any introduced NaNs are filled with 0.0

    # Ensure all required types (columns) are present and ordered. 'types' is from the global definition in cell 750a504b.
    current_columns = set(final_df.columns)
    missing_columns = list(set(types) - current_columns)
    for col in missing_columns:
        final_df[col] = 0.0

    # Ensure all required countries (index) are present and reindex
    final_df = final_df.reindex(index=all_countries_in_gipt, fill_value=0.0)

    # Select and order the desired technology columns. Exclude 'solar dist' if not needed in final output.
    final_output_columns = ['coal', 'oil/gas', 'utility-scale solar', 'wind', 'nuclear', 'hydropower', 'bioenergy', 'geothermal']
    final_df = final_df[final_output_columns]

    # Add region and sub-region information from the main gipt DataFrame
    country_info = gipt[['Country/area', 'Region', 'Subregion']].drop_duplicates(subset="Country/area").set_index('Country/area')
    final_df['sub-region'] = final_df.index.map(country_info['Subregion'].get)
    final_df['region'] = final_df.index.map(country_info['Region'].get)

    # Reorder columns to match the desired output format: region, sub-region, Country/area, then technologies
    final_df = final_df.reset_index().rename(columns={'index': 'Country/area'})
    final_df = final_df[['region', 'sub-region', 'Country/area'] + final_output_columns]

    return final_df

# Test the function:
#cap_by_country_and_tech('operating')[:200]

Type,region,sub-region,Country/area,coal,oil/gas,utility-scale solar,wind,nuclear,hydropower,bioenergy,geothermal
0,Asia,Southern Asia,Afghanistan,0.0,266.60,80.312112,0.0,0.0,354.0,0.0,0.0
1,Europe,Southern Europe,Albania,0.0,0.00,320.433176,0.0,0.0,1770.0,0.0,0.0
2,Africa,Northern Africa,Algeria,0.0,27580.60,420.001250,10.0,0.0,196.0,0.0,0.0
3,Oceania,Polynesia,American Samoa,0.0,23.00,1.309000,0.0,0.0,0.0,0.0,0.0
4,Europe,Southern Europe,Andorra,0.0,0.00,3.300000,0.0,0.0,45.0,18.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
195,Asia,Western Asia,Syria,0.0,7662.30,188.488571,0.0,0.0,1585.0,0.0,0.0
196,Asia,Eastern Asia,Taiwan,14999.0,23122.65,4594.689500,4396.2,0.0,4480.0,502.8,5.2
197,Asia,Central Asia,Tajikistan,400.0,206.00,1.200000,0.0,0.0,5834.0,0.0,0.0
198,Africa,Sub-Saharan Africa,Tanzania,0.0,849.00,139.900000,0.0,0.0,2683.0,46.0,0.0


In [ ]:
##########################################################################################################################################################
# 2) RUN THE FUNCTION PER TAB IN TARGET SUMMARY TABLE: https://docs.google.com/spreadsheets/d/1a1s9hHB9bw-WX_RgUDseTwq6ALqu3M_4P7iMkKHxlEA/edit#gid=42360804
###########################################################################################################################################################

#AUTHORIZE EDITS
gc = pygsheets.authorize(service_file='/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Work in Progress/GDRIVE_API_CREDENTIALS.json')
spreadsheet = gc.open_by_key("1a1s9hHB9bw-WX_RgUDseTwq6ALqu3M_4P7iMkKHxlEA")
#ANNOUNCED TAB
test = spreadsheet.worksheet('title', 'Announced')
test.set_dataframe(cap_by_country_and_tech('announced'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Pre-construction TAB
pre_construction=cap_by_country_and_tech('pre-construction')
test = spreadsheet.worksheet('title', 'Pre-construction')
test.set_dataframe(pre_construction, 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Construction TAB
test = spreadsheet.worksheet('title', 'Construction')
test.set_dataframe(cap_by_country_and_tech('construction'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Shelved TAB
test = spreadsheet.worksheet('title', 'Shelved')
test.set_dataframe(cap_by_country_and_tech('shelved'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Operating TAB
test = spreadsheet.worksheet('title', 'Operating')
test.set_dataframe(cap_by_country_and_tech('operating'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Mothballed TAB
test = spreadsheet.worksheet('title', 'Mothballed')
test.set_dataframe(cap_by_country_and_tech('mothballed'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Cancelled TAB
test = spreadsheet.worksheet('title', 'Cancelled')
test.set_dataframe(cap_by_country_and_tech('cancelled'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Retired TAB
test = spreadsheet.worksheet('title', 'Retired')
test.set_dataframe(cap_by_country_and_tech('retired'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)


In [ ]:
##########################################################################################################################################
# REFINED FUNCTION TO HANDLE AGGREGATION BY: TECHNOLOGY, STATUS AND REGION (OR CUSTOM GROUPING)
# This function replaces and consolidates the logic from both `cap_by_region_and_status` and `cap_by_region_and_status_custom`
##########################################################################################################################################

techs=['coal','oil/gas','utility-scale solar','wind','nuclear','hydropower','bioenergy','geothermal']
all_status=['announced','pre-permit','permitted','pre-construction','construction','shelved','operating','mothballed','cancelled','retired']

def capacity_by_tech_and_status(area_name, area_type='Region', dataframe=None, custom_definitions=None):
    """
    Calculates power capacity by technology and status for a given area (region, subregion, or custom country list).

    Args:
        area_name (str): The name of the area to filter by (e.g., 'Africa', 'EU27', 'Global').
        area_type (str): The type of area. Can be 'Region', 'Subregion', 'Custom', or 'Global'.
                         - 'Region' or 'Subregion': Filters `gipt_hydro_adjust` by the corresponding column.
                         - 'Custom': Filters `gipt_hydro_adjust` by countries defined in `custom_definitions[area_name]`.
                         - 'Global': Aggregates all data.
        dataframe (pd.DataFrame, optional): The input GIPT DataFrame (gipt_hydro_adjust). Defaults to global `gipt_hydro_adjust`.
        custom_definitions (pd.DataFrame, optional): DataFrame containing custom region definitions (e.g., `defs`).
                                                   Required if `area_type` is 'Custom'.

    Returns:
        pd.DataFrame: A DataFrame with power capacity aggregated by technology and status for the specified area.
    """
    df_to_use = dataframe if dataframe is not None else gipt.copy()

    # Apply filtering based on area_type
    if area_type == 'Global':
        filtered_df = df_to_use
    elif area_type == 'Custom':
        if custom_definitions is None:
            raise ValueError("custom_definitions DataFrame is required when area_type is 'Custom'.")
        custom_countries = custom_definitions[custom_definitions[area_name] == 1]['GEM Standard Country Name/Area']
        filtered_df = df_to_use[df_to_use['Country/area'].isin(custom_countries)]
    elif area_type == 'Region':
        filtered_df = df_to_use[df_to_use['Region'] == area_name]
    elif area_type == 'Subregion':
        filtered_df = df_to_use[df_to_use['Subregion'] == area_name]
    else:
        raise ValueError(f"Invalid area_type: {area_type}. Must be 'Region', 'Subregion', 'Custom', or 'Global'.")

    # Group by Type and Status, then unstack to get technologies as index and statuses as columns
    grouped_df = filtered_df.groupby(['Type', 'Status'])['Capacity (MW)'].sum().unstack(fill_value=0.0)

    # Reindex to ensure all technologies and statuses are present and ordered
    # Ensure all 'techs' are present as index
    grouped_df = grouped_df.reindex(techs, fill_value=0.0)

    # Sum 'pre-permit' and 'permitted' into 'pre-construction'
    if 'pre-permit' in grouped_df.columns:
        grouped_df['pre-construction'] = grouped_df['pre-construction'] + grouped_df['pre-permit']
    if 'permitted' in grouped_df.columns:
        grouped_df['pre-construction'] = grouped_df['pre-construction'] + grouped_df['permitted']

    # Ensure all 'all_status' columns are present and ordered. Note: 'pre-permit' and 'permitted' are now implicitly handled.
    final_status_columns = ['announced', 'pre-construction', 'construction', 'shelved', 'operating', 'mothballed', 'cancelled', 'retired']

    current_statuses = set(grouped_df.columns)
    missing_statuses = list(set(final_status_columns) - current_statuses)
    for s in missing_statuses:
        grouped_df[s] = 0.0

    final_df_output = grouped_df[final_status_columns]

    final_df_output.index.name = area_name # Set index name to the area_name
    return final_df_output

# Test the refined function for Global
#display(capacity_by_tech_and_status('Global', area_type='Global'))

# Test for a specific Region
#display(capacity_by_tech_and_status('Northern Africa', area_type='Subregion'))

# Test for a custom region (e.g., 'EU27')
# display(capacity_by_tech_and_status('EU27', area_type='Custom', custom_definitions=defs))

In [ ]:
##########################################################################################################################################
# RUN THE FUNCTION PER TAB IN TARGET SUMMARY TABLE: https://docs.google.com/spreadsheets/d/1tHwx9MRi7WhyzgqR9oZD78f2vMyFpLpZ0zcxcsmT-Io/edit#gid=1489499360
##########################################################################################################################################

#AUTHORIZE EDITS
gc = pygsheets.authorize(service_file='/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Work in Progress/GDRIVE_API_CREDENTIALS.json')
spreadsheet = gc.open_by_key("1tHwx9MRi7WhyzgqR9oZD78f2vMyFpLpZ0zcxcsmT-Io")
#Global TAB
test = spreadsheet.worksheet('title', 'Global')
test.set_dataframe(capacity_by_tech_and_status('Global', area_type='Global'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)

#AFRICA TAB
cap_tech_status_REGION=capacity_by_tech_and_status('Africa', area_type='Region').fillna(0)
test = spreadsheet.worksheet('title', 'Africa')
test.set_dataframe(cap_tech_status_REGION, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Northern Africa', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Africa')
test.set_dataframe(cap_tech_status_REGION, 'B16', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Sub-Saharan Africa', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Africa')
test.set_dataframe(cap_tech_status_REGION, 'B26', copy_index=False, copy_head=False, extend=False, fit=False)

#AMERICAS TAB
cap_tech_status_REGION=capacity_by_tech_and_status('Americas', area_type='Region').fillna(0)
test = spreadsheet.worksheet('title', 'Americas')
test.set_dataframe(cap_tech_status_REGION, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Latin America and the Caribbean', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Americas')
test.set_dataframe(cap_tech_status_REGION, 'B16', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Northern America', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Americas')
test.set_dataframe(cap_tech_status_REGION, 'B26', copy_index=False, copy_head=False, extend=False, fit=False)

#ASIA TAB
cap_tech_status_REGION=capacity_by_tech_and_status('Asia', area_type='Region').fillna(0)
test = spreadsheet.worksheet('title', 'Asia')
test.set_dataframe(cap_tech_status_REGION, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Central Asia', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Asia')
test.set_dataframe(cap_tech_status_REGION, 'B16', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Eastern Asia', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Asia')
test.set_dataframe(cap_tech_status_REGION, 'B26', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('South-eastern Asia', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Asia')
test.set_dataframe(cap_tech_status_REGION, 'B36', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Southern Asia', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Asia')
test.set_dataframe(cap_tech_status_REGION, 'B46', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Western Asia', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Asia')
test.set_dataframe(cap_tech_status_REGION, 'B56', copy_index=False, copy_head=False, extend=False, fit=False)

#EUROPE TAB
cap_tech_status_REGION=capacity_by_tech_and_status('Europe', area_type='Region').fillna(0)
test = spreadsheet.worksheet('title', 'Europe')
test.set_dataframe(cap_tech_status_REGION, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Eastern Europe', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Europe')
test.set_dataframe(cap_tech_status_REGION, 'B16', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Northern Europe', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Europe')
test.set_dataframe(cap_tech_status_REGION, 'B26', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Southern Europe', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Europe')
test.set_dataframe(cap_tech_status_REGION, 'B36', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Western Europe', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Europe')
test.set_dataframe(cap_tech_status_REGION, 'B46', copy_index=False, copy_head=False, extend=False, fit=False)

#OCEANIA TAB
cap_tech_status_REGION=capacity_by_tech_and_status('Oceania', area_type='Region').fillna(0)
test = spreadsheet.worksheet('title', 'Oceania')
test.set_dataframe(cap_tech_status_REGION, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Australia and New Zealand', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Oceania')
test.set_dataframe(cap_tech_status_REGION, 'B16', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Melanesia', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Oceania')
test.set_dataframe(cap_tech_status_REGION, 'B26', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Micronesia', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Oceania')
test.set_dataframe(cap_tech_status_REGION, 'B36', copy_index=False, copy_head=False, extend=False, fit=False)
#
cap_tech_status_REGION=capacity_by_tech_and_status('Polynesia', area_type='Subregion').fillna(0)
test = spreadsheet.worksheet('title', 'Oceania')
test.set_dataframe(cap_tech_status_REGION, 'B46', copy_index=False, copy_head=False, extend=False, fit=False)
defs
####CUSTOM REGIONS
#G7
cap_tech_status_REGION=capacity_by_tech_and_status('G7 (including EU27)', area_type='Custom', custom_definitions=defs).fillna(0)
test = spreadsheet.worksheet('title', 'G7 (including EU27)')
test.set_dataframe(cap_tech_status_REGION, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#G20
cap_tech_status_REGION=capacity_by_tech_and_status('G20 (including EU27 and African Union)', area_type='Custom', custom_definitions=defs).fillna(0)
test = spreadsheet.worksheet('title', 'G20 (including EU27+African Union)')
test.set_dataframe(cap_tech_status_REGION, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#EU27
cap_tech_status_REGION=capacity_by_tech_and_status('EU27', area_type='Custom', custom_definitions=defs).fillna(0)
test = spreadsheet.worksheet('title', 'EU27')
test.set_dataframe(cap_tech_status_REGION, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#African Union
cap_tech_status_REGION=capacity_by_tech_and_status('African Union', area_type='Custom', custom_definitions=defs).fillna(0)
test = spreadsheet.worksheet('title', 'African Union')
test.set_dataframe(cap_tech_status_REGION, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#BRICS
cap_tech_status_REGION=capacity_by_tech_and_status('BRICS+ (Brazil, Russia, India, China, South Africa, Ethiopia, the United Arab Emirates, Iran, Egypt, and Indonesia)', area_type='Custom', custom_definitions=defs).fillna(0)
test = spreadsheet.worksheet('title', 'BRICS')
test.set_dataframe(cap_tech_status_REGION, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#OECD
cap_tech_status_REGION=capacity_by_tech_and_status('OECD', area_type='Custom', custom_definitions=defs).fillna(0)
test = spreadsheet.worksheet('title', 'OECD')
test.set_dataframe(cap_tech_status_REGION, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Non-OECD (exc. China)
cap_tech_status_REGION=capacity_by_tech_and_status('Non-OECD (exc. China)', area_type='Custom', custom_definitions=defs).fillna(0)
test = spreadsheet.worksheet('title', 'Non-OECD (exc. China)')
test.set_dataframe(cap_tech_status_REGION, 'B6', copy_index=False, copy_head=False, extend=False, fit=False)



In [ ]:
##########################################################################################################################################
# A FUNCTION TO HANDLE AGGREGATION BY: REGION and STATUS, GROUPED BY TECHNOLOGY
# This updates the sheet called: "Power capacity by region and status, grouped by technology" https://docs.google.com/spreadsheets/d/1fHZ2h47iqyy3tywtajQXNfMH0lJ4J5znVG3OxLL2JK0/edit#gid=1953853820
##########################################################################################################################################

regs1=['Africa','Northern Africa','Sub-Saharan Africa','Americas','Latin America and the Caribbean','Northern America','Asia','Central Asia','Eastern Asia','South-eastern Asia','Southern Asia','Western Asia',
'Europe','Eastern Europe','Northern Europe','Southern Europe','Western Europe','Oceania','Australia and New Zealand','Melanesia','Micronesia','Polynesia']
types=['coal', 'oil/gas', 'utility-scale solar', 'wind', 'nuclear','hydropower', 'bioenergy','geothermal']
status=['announced','pre-permit','permitted','pre-construction','construction','shelved','operating','mothballed','cancelled','retired','inactive']

mononational=gipt[(gipt['Country/area 2 (hydropower only)'].isnull())]
binational1=gipt[(gipt['Country/area 2 (hydropower only)'].notnull())][['Country/area 2 (hydropower only)','Country/area 2 Capacity (MW) (hydropower only)','Type','Status','Region','Subregion']]
binational2=gipt[(gipt['Country/area 2 (hydropower only)'].notnull())][['Country/area 1 (hydropower only)','Country/area 1 Capacity (MW) (hydropower only)','Type','Status','Region','Subregion']]
binational1.columns=['Country/area','Capacity (MW)', 'Type','Status','Region','Subregion']
binational2.columns=['Country/area','Capacity (MW)', 'Type','Status','Region','Subregion']
binational=pandas.concat([binational1,binational2],axis=0)
mononational[['Country/area','Capacity (MW)', 'Type','Region','Subregion','Status']]

gipt_hydro_adjust=pandas.concat([mononational[['Country/area','Capacity (MW)', 'Type','Region','Subregion','Status']],binational[['Country/area','Capacity (MW)', 'Type','Region','Subregion','Status']]])

def cap_by_region_and_status(ttype):
    res=[]
    res.append(gipt_hydro_adjust[(gipt_hydro_adjust.Type==ttype)].groupby(['Region','Status'])['Capacity (MW)'].sum().unstack())
    res.append(gipt_hydro_adjust[(gipt_hydro_adjust.Type==ttype)].groupby(['Subregion','Status'])['Capacity (MW)'].sum().unstack())
    tmp=pandas.concat(res,axis=0)
    missing=list(set(status) - set(list(tmp.columns)))
    for i in missing:
        aa=pandas.DataFrame(index=list(tmp.index),columns=[i]).fillna(0.0)
        tmp=pandas.concat([tmp,aa],axis=1)
    missing=list(set(regs1) - set(list(tmp.index)))
    for i in missing:
        aa=pandas.DataFrame(index=[i],columns=tmp.columns).fillna(0.0)
        tmp=pandas.concat([tmp,aa],axis=0)
    tmp.index.name=ttype
    tmp=tmp.loc[regs1].fillna(0)
    # if ttype=='coal':
    #   tmp['pre-construction']=(tmp['pre-permit']+tmp['permitted']).values
    tmp=tmp[['announced','pre-construction','construction','shelved','operating','mothballed','cancelled','retired']]
    return tmp

#test it:
cap_by_region_and_status('oil/gas')



/tmp/ipykernel_15664/3661928689.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  aa=pandas.DataFrame(index=list(tmp.index),columns=[i]).fillna(0.0)
/tmp/ipykernel_15664/3661928689.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  aa=pandas.DataFrame(index=list(tmp.index),columns=[i]).fillna(0.0)
/tmp/ipykernel_15664/3661928689.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set

,announced,pre-construction,construction,shelved,operating,mothballed,cancelled,retired
oil/gas,,,,,,,,
Africa,24958.00,19324.000,15566.70,22887.00,135654.30,1660.0,13164.0,1215.9
Northern Africa,6675.00,2015.000,13295.00,5731.00,105646.50,760.0,6351.0,1067.0
Sub-Saharan Africa,18283.00,17309.000,2271.70,17156.00,30007.80,900.0,6813.0,148.9
Americas,117477.60,180003.398,43196.85,47072.20,740860.00,24662.1,67784.2,19445.3
Latin America and the Caribbean,45590.00,13034.500,12614.25,31928.90,146594.30,16310.4,37698.3,1691.3
Northern America,71887.60,166968.898,30582.60,15143.30,594265.70,8351.7,30085.9,17754.0
Asia,168774.04,234418.400,140953.90,52183.00,967799.58,14103.1,189846.0,35656.0
Central Asia,2980.04,6698.000,11052.50,2452.00,26804.75,0.0,1336.0,720.0
Eastern Asia,88539.50,86934.000,43339.80,11315.00,358716.19,1900.0,46277.0,23196.0


In [ ]:
#####################################################################
# RUN THE FUNCTION PER TAB IN TARGET SUMMARY TABLE: https://docs.google.com/spreadsheets/d/1fHZ2h47iqyy3tywtajQXNfMH0lJ4J5znVG3OxLL2JK0/edit#gid=1953853820
#####################################################################

#AUTHORIZE EDITS
gc = pygsheets.authorize(service_file='/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Work in Progress/GDRIVE_API_CREDENTIALS.json')
spreadsheet = gc.open_by_key("1fHZ2h47iqyy3tywtajQXNfMH0lJ4J5znVG3OxLL2JK0")
#Coal TAB
test = spreadsheet.worksheet('title', 'Coal')
test.set_dataframe(cap_by_region_and_status('coal'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Oil/gas TAB
test = spreadsheet.worksheet('title', 'Oil and gas')
test.set_dataframe(cap_by_region_and_status('oil/gas'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Solar TAB
test = spreadsheet.worksheet('title', 'Utility-scale solar')
test.set_dataframe(cap_by_region_and_status('utility-scale solar'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Wind TAB
test = spreadsheet.worksheet('title', 'Wind')
test.set_dataframe(cap_by_region_and_status('wind'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Nuclear TAB
test = spreadsheet.worksheet('title', 'Nuclear')
test.set_dataframe(cap_by_region_and_status('nuclear'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Hydro TAB
test = spreadsheet.worksheet('title', 'Hydropower')
test.set_dataframe(cap_by_region_and_status('hydropower'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Bio TAB
test = spreadsheet.worksheet('title', 'Bioenergy')
test.set_dataframe(cap_by_region_and_status('bioenergy'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Geo TAB
test = spreadsheet.worksheet('title', 'Geothermal')
test.set_dataframe(cap_by_region_and_status('geothermal'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)


/tmp/ipykernel_273/3661928689.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  aa=pandas.DataFrame(index=list(tmp.index),columns=[i]).fillna(0.0)
/tmp/ipykernel_273/3661928689.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  aa=pandas.DataFrame(index=list(tmp.index),columns=[i]).fillna(0.0)
/tmp/ipykernel_273/3661928689.py:28: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_optio

In [ ]:
##########################################################################################################################################
# A FUNCTION TO HANDLE AGGREGATION BY: Country and STATUS, GROUPED BY TECHNOLOGY
# This updates the sheet called: "Power capacity by region and status, grouped by technology" https://docs.google.com/spreadsheets/d/1XS9kjfssMYFqQ7uELf90YvDidUIYpHoCxuTVvCZZjZA/edit#gid=2082689497
##########################################################################################################################################

countries1=np.sort(gipt_hydro_adjust['Country/area'].unique())
ttype='coal'
def cap_by_country_and_status(ttype):
    res=[]
    res.append(gipt_hydro_adjust[(gipt_hydro_adjust.Type==ttype)].groupby(['Country/area','Status'])['Capacity (MW)'].sum().unstack())
    tmp=pandas.concat(res,axis=0)
    missing=list(set(status) - set(list(tmp.columns)))
    for i in missing:
        aa=pandas.DataFrame(index=list(tmp.index),columns=[i]).fillna(0.0)
        tmp=pandas.concat([tmp,aa],axis=1)
    missing=list(set(countries1) - set(list(tmp.index)))
    for i in missing:
        aa=pandas.DataFrame(index=[i],columns=tmp.columns).fillna(0.0)
        tmp=pandas.concat([tmp,aa],axis=0)
    #tmp.index.name=ttype
    tmp=tmp.sort_index().fillna(0)
    tmp['sub-region']=gipt.drop_duplicates(subset="Country/area", keep="first")[['Country/area','Region','Subregion']].set_index('Country/area').loc[tmp.index.values]['Subregion'].values
    tmp['region']=gipt.drop_duplicates(subset="Country/area", keep="first")[['Country/area','Region','Subregion']].set_index('Country/area').loc[tmp.index.values]['Region'].values
    # if ttype=='coal':
    #   tmp['pre-construction']=(tmp['pre-permit']+tmp['permitted']).values
    tmp=tmp.reset_index()[['region','sub-region','index','announced','pre-construction','construction','shelved','operating','mothballed','cancelled','retired']]
    return tmp

#Test the function:
cap_by_country_and_status('utility-scale solar')[:50]


/tmp/ipykernel_15664/749388461.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  aa=pandas.DataFrame(index=list(tmp.index),columns=[i]).fillna(0.0)
/tmp/ipykernel_15664/749388461.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  aa=pandas.DataFrame(index=list(tmp.index),columns=[i]).fillna(0.0)
/tmp/ipykernel_15664/749388461.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_op

,region,sub-region,index,announced,pre-construction,construction,shelved,operating,mothballed,cancelled,retired
0,Asia,Southern Asia,Afghanistan,0.000000,0.000000,77.844039,2838.302900,80.312112,0.000000,2312.250118,0.000000
1,Europe,Southern Europe,Albania,291.319243,348.262514,87.000000,682.195491,320.433176,0.000000,100.000000,0.000000
2,Africa,Northern Africa,Algeria,45.531250,1684.871250,2255.315000,22.359375,420.001250,0.000000,4004.836875,0.000000
3,Oceania,Polynesia,American Samoa,0.000000,0.000000,18.700000,0.000000,1.309000,0.000000,0.000000,0.000000
4,Europe,Southern Europe,Andorra,0.000000,0.000000,0.000000,0.000000,3.300000,0.000000,0.000000,0.000000
5,Africa,Sub-Saharan Africa,Angola,626.316667,78.300000,21.750000,111.099000,488.873067,0.000000,0.000000,0.000000
6,Americas,Latin America and the Caribbean,Anguilla,0.000000,0.000000,0.000000,0.000000,1.200000,0.000000,0.000000,0.000000
7,Americas,Latin America and the Caribbean,Antigua and Barbuda,0.000000,0.000000,0.000000,0.000000,10.300000,0.000000,0.000000,0.000000
8,Americas,Latin America and the Caribbean,Argentina,4355.633261,271.210826,427.064130,68.026087,2764.085717,0.000000,287.826087,0.000000
9,Asia,Western Asia,Armenia,200.000000,0.000000,230.859357,0.000000,263.099571,0.000000,0.000000,0.000000


In [ ]:
#####################################################################
# RUN THE FUNCTION PER TAB IN TARGET SUMMARY TABLE: https://docs.google.com/spreadsheets/d/1XS9kjfssMYFqQ7uELf90YvDidUIYpHoCxuTVvCZZjZA/edit#gid=2082689497
#####################################################################

#AUTHORIZE EDITS
gc = pygsheets.authorize(service_file='/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Work in Progress/GDRIVE_API_CREDENTIALS.json')
spreadsheet = gc.open_by_key("1XS9kjfssMYFqQ7uELf90YvDidUIYpHoCxuTVvCZZjZA")
#Coal TAB
test = spreadsheet.worksheet('title', 'Coal')
test.set_dataframe(cap_by_country_and_status('coal'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Oil/gas TAB
test = spreadsheet.worksheet('title', 'Oil and gas')
test.set_dataframe(cap_by_country_and_status('oil/gas'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Solar TAB
test = spreadsheet.worksheet('title', 'Utility-scale solar')
test.set_dataframe(cap_by_country_and_status('utility-scale solar'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Wind TAB
test = spreadsheet.worksheet('title', 'Wind')
test.set_dataframe(cap_by_country_and_status('wind'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Nuclear TAB
test = spreadsheet.worksheet('title', 'Nuclear')
test.set_dataframe(cap_by_country_and_status('nuclear'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Hydro TAB
test = spreadsheet.worksheet('title', 'Hydropower')
test.set_dataframe(cap_by_country_and_status('hydropower'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Bio TAB
test = spreadsheet.worksheet('title', 'Bioenergy')
test.set_dataframe(cap_by_country_and_status('bioenergy'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Geo TAB
test = spreadsheet.worksheet('title', 'Geothermal')
test.set_dataframe(cap_by_country_and_status('geothermal'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)



/tmp/ipykernel_15664/749388461.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  aa=pandas.DataFrame(index=list(tmp.index),columns=[i]).fillna(0.0)
/tmp/ipykernel_15664/749388461.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  aa=pandas.DataFrame(index=list(tmp.index),columns=[i]).fillna(0.0)
/tmp/ipykernel_15664/749388461.py:14: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_op

In [31]:
####################################################################################################################
# A FUNCTION TO HANDLE AGGREGATION BY: REGION, YEAR, FOR OPERATING PLANTS  https://docs.google.com/spreadsheets/d/1mKNIvxmW3wBMX-0OT-BUGX2F_KqEqsyW0D0jLCmyzAg/edit?gid=1953853820#gid=1953853820
# This updates the sheet called: "Power capacity by region and year, grouped by status"
####################################################################################################################

def cap_by_region_and_year(tech, year_min=2000, year_max=2025, dataframe=None):
    """
    Calculates operating power capacity by region/subregion and Start year for a specific technology type.
    Handles bi-national hydropower projects by splitting their capacity between relevant countries.

    Args:
        tech (str): The technology type to filter by (e.g., 'coal', 'hydropower').
        year_min (int): The minimum Start year to include (inclusive). Defaults to 2000.
        year_max (int): The maximum Start year to include (inclusive). Defaults to 2025.
        dataframe (pd.DataFrame, optional): The input GIPT DataFrame. If None,
                                           it uses the global `gipt` DataFrame.

    Returns:
        pd.DataFrame: A DataFrame with operating capacity aggregated by region/subregion
                      and Start year for the specified technology, indexed by 'Region/Subregion'.
                      Columns represent years from `year_min` to `year_max` and 'Capacity (MW) no year'.
    """
    if dataframe is None:
        df = gipt.copy()
    else:
        df = dataframe.copy()

    # Filter projects for 'operating' status and the specific technology type
    df_filtered_status_and_tech = df[(df.Status == 'operating') & (df.Type == tech)].copy()

    # Convert 'Start year' to numeric, coercing errors to NaN
    df_filtered_status_and_tech['Start year'] = pd.to_numeric(
        df_filtered_status_and_tech['Start year'], errors='coerce'
    )

    # Separate projects with missing 'Start year'
    missing_year_projects = df_filtered_status_and_tech[df_filtered_status_and_tech['Start year'].isna()].copy()
    # Filter out missing year projects from the main DataFrame for year-wise aggregation
    df_valid_years = df_filtered_status_and_tech[df_filtered_status_and_tech['Start year'].notna()].copy()
    df_valid_years['Start year'] = df_valid_years['Start year'].astype(int)

    # Further filter by the specified year range for projects with valid years
    df_valid_years = df_valid_years[
        (df_valid_years['Start year'] >= year_min) &
        (df_valid_years['Start year'] <= year_max)
    ]

    if tech == 'hydropower':
        # --- Hydropower-specific handling for bi-national projects (valid years) ---
        mononational_hydro_valid_years = df_valid_years[
            df_valid_years['Country/area 2 (hydropower only)'].isnull()
        ]
        binational_hydro_valid_years = df_valid_years[
            df_valid_years['Country/area 2 (hydropower only)'].notnull()
        ]

        binational_hydro_part1_valid_years = binational_hydro_valid_years[[
            'Country/area 2 (hydropower only)', 'Country/area 2 Capacity (MW) (hydropower only)',
            'Region', 'Subregion', 'Start year'
        ]].rename(columns={
            'Country/area 2 (hydropower only)': 'Country/area_dummy',
            'Country/area 2 Capacity (MW) (hydropower only)': 'Capacity (MW)'
        })

        binational_hydro_part2_valid_years = binational_hydro_valid_years[[
            'Country/area 1 (hydropower only)', 'Country/area 1 Capacity (MW) (hydropower only)',
            'Region', 'Subregion', 'Start year'
        ]].rename(columns={
            'Country/area 1 (hydropower only)': 'Country/area_dummy',
            'Country/area 1 Capacity (MW) (hydropower only)': 'Capacity (MW)'
        })

        all_hydro_data_valid_years = pd.concat([
            mononational_hydro_valid_years[['Region', 'Subregion', 'Capacity (MW)', 'Start year']],
            binational_hydro_part1_valid_years[['Region', 'Subregion', 'Capacity (MW)', 'Start year']],
            binational_hydro_part2_valid_years[['Region', 'Subregion', 'Capacity (MW)', 'Start year']]
        ])

        agg_region = all_hydro_data_valid_years.groupby(['Region', 'Start year'])['Capacity (MW)'].sum().unstack(fill_value=0.0)
        agg_subregion = all_hydro_data_valid_years.groupby(['Subregion', 'Start year'])['Capacity (MW)'].sum().unstack(fill_value=0.0)

        # --- Hydropower-specific handling for bi-national projects (missing years) ---
        mononational_hydro_missing_year = missing_year_projects[
            missing_year_projects['Country/area 2 (hydropower only)'].isnull()
        ].copy()
        binational_hydro_missing_year = missing_year_projects[
            missing_year_projects['Country/area 2 (hydropower only)'].notnull()
        ].copy()

        binational_hydro_part1_missing_year = binational_hydro_missing_year[[
            'Country/area 2 (hydropower only)', 'Country/area 2 Capacity (MW) (hydropower only)',
            'Region', 'Subregion'
        ]].rename(columns={
            'Country/area 2 (hydropower only)': 'Country/area_dummy',
            'Country/area 2 Capacity (MW) (hydropower only)': 'Capacity (MW)'
        })

        binational_hydro_part2_missing_year = binational_hydro_missing_year[[
            'Country/area 1 (hydropower only)', 'Country/area 1 Capacity (MW) (hydropower only)',
            'Region', 'Subregion'
        ]].rename(columns={
            'Country/area 1 (hydropower only)': 'Country/area_dummy',
            'Country/area 1 Capacity (MW) (hydropower only)': 'Capacity (MW)'
        })

        all_hydro_data_missing_year = pd.concat([
            mononational_hydro_missing_year[['Region', 'Subregion', 'Capacity (MW)']],
            binational_hydro_part1_missing_year[['Region', 'Subregion', 'Capacity (MW)']],
            binational_hydro_part2_missing_year[['Region', 'Subregion', 'Capacity (MW)']]
        ])

        missing_year_agg_region = all_hydro_data_missing_year.groupby('Region')['Capacity (MW)'].sum()
        missing_year_agg_subregion = all_hydro_data_missing_year.groupby('Subregion')['Capacity (MW)'].sum()

    else:
        # --- Standard handling for all other technologies (non-hydropower) ---
        agg_region = df_valid_years.groupby(['Region', 'Start year'])['Capacity (MW)'].sum().unstack(fill_value=0.0)
        agg_subregion = df_valid_years.groupby(['Subregion', 'Start year'])['Capacity (MW)'].sum().unstack(fill_value=0.0)

        missing_year_agg_region = missing_year_projects.groupby('Region')['Capacity (MW)'].sum()
        missing_year_agg_subregion = missing_year_projects.groupby('Subregion')['Capacity (MW)'].sum()

    # Combine region and subregion aggregations into a single DataFrame for valid years
    combined_agg_df = pd.concat([agg_region, agg_subregion], axis=0)

    # Combine region and subregion aggregations for missing years
    combined_missing_year_agg = pd.concat([missing_year_agg_region, missing_year_agg_subregion], axis=0)
    # Ensure the missing year series has the same index as regs1
    missing_year_series = pd.Series(0.0, index=regs1, name='Capacity (MW) no year')
    missing_year_series.update(combined_missing_year_agg)

    # Use the helper function `_ensure_dimensions` to ensure all standard regions/subregions
    # and all years in the range are present and correctly ordered in the output DataFrame.
    final_df_result = _ensure_dimensions(
        combined_agg_df, regs1, list(range(year_min, year_max + 1)), index_name='Region/Subregion'
    )
    final_df_result.columns.name = 'Start year' # Name the columns index for clarity

    # Add the 'Capacity (MW) no year' column
    final_df_result['Capacity (MW) no year'] = missing_year_series.loc[final_df_result.index].fillna(0.0).values

    return final_df_result

# Test the new refined function for 'coal' technology for years 2000-2025
cap_by_region_and_year('hydropower', year_min=2000, year_max=2025)


Start year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Capacity (MW) no year
Region/Subregion,,,,,,,,,,,,,,,,,,,,,
Africa,390.0,73.0,200.0,292.0,1170.0,0.0,0.0,0.0,124.0,1600.0,...,3045.0,577.0,394.0,744.0,834.0,840.0,950.0,2577.0,5645.0,124.0
Northern Africa,0.0,0.0,0.0,92.0,466.0,0.0,0.0,0.0,64.0,1250.0,...,0.0,352.0,0.0,0.0,0.0,0.0,0.0,350.0,0.0,0.0
Sub-Saharan Africa,390.0,73.0,200.0,200.0,704.0,0.0,0.0,0.0,60.0,350.0,...,3045.0,225.0,394.0,744.0,834.0,840.0,950.0,2227.0,5645.0,124.0
Americas,2579.0,1914.0,3324.0,4322.0,1448.0,2002.0,1707.0,3364.0,688.0,1022.0,...,1578.0,584.0,2264.0,824.0,578.0,1726.0,67.0,153.0,1312.0,743.0
Latin America and the Caribbean,2579.0,1790.0,2811.0,3113.0,1168.0,1476.0,1707.0,2283.0,550.0,939.0,...,1026.0,584.0,1431.0,0.0,523.0,1481.0,67.0,153.0,212.0,743.0
Northern America,0.0,124.0,513.0,1209.0,280.0,526.0,0.0,1081.0,138.0,83.0,...,552.0,0.0,833.0,824.0,55.0,245.0,0.0,0.0,1100.0,0.0
Asia,12242.0,8192.0,3268.0,7520.0,7711.0,9803.0,13445.0,11862.0,18008.0,22154.0,...,13469.0,10704.0,9186.0,10658.0,25628.0,25405.0,15985.0,21196.0,12931.0,3821.0
Central Asia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,670.0,0.0,...,0.0,1200.0,0.0,0.0,75.0,0.0,175.0,25.0,0.0,0.0
Eastern Asia,10420.0,4032.0,642.0,3245.0,4819.0,7387.0,7996.0,8107.0,13594.0,19785.0,...,10060.0,4630.0,5319.0,5175.0,23494.0,23215.0,14304.0,19181.0,6580.0,3554.0


In [27]:
#####################################################################
# RUN THE FUNCTION PER TAB IN TARGET SUMMARY TABLE: https://docs.google.com/spreadsheets/d/1mKNIvxmW3wBMX-0OT-BUGX2F_KqEqsyW0D0jLCmyzAg/edit?gid=1953853820#gid=1953853820
#####################################################################

#AUTHORIZE EDITS
gc = pygsheets.authorize(service_file='/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Work in Progress/GDRIVE_API_CREDENTIALS.json')
spreadsheet = gc.open_by_key("1mKNIvxmW3wBMX-0OT-BUGX2F_KqEqsyW0D0jLCmyzAg")
#Coal TAB
test = spreadsheet.worksheet('title', 'Coal')
test.set_dataframe(cap_by_region_and_year('coal'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Oil/gas TAB
test = spreadsheet.worksheet('title', 'Oil and gas')
test.set_dataframe(cap_by_region_and_year('oil/gas'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Solar TAB
test = spreadsheet.worksheet('title', 'Utility-scale solar')
test.set_dataframe(cap_by_region_and_year('utility-scale solar'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Wind TAB
test = spreadsheet.worksheet('title', 'Wind')
test.set_dataframe(cap_by_region_and_year('wind'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Nuclear TAB
test = spreadsheet.worksheet('title', 'Nuclear')
test.set_dataframe(cap_by_region_and_year('nuclear'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Hydro TAB
test = spreadsheet.worksheet('title', 'Hydropower')
test.set_dataframe(cap_by_region_and_year('hydropower'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Bio TAB
test = spreadsheet.worksheet('title', 'Bioenergy')
test.set_dataframe(cap_by_region_and_year('bioenergy'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)
#Geo TAB
test = spreadsheet.worksheet('title', 'Geothermal')
test.set_dataframe(cap_by_region_and_year('geothermal'), 'B6', copy_index=False, copy_head=False, extend=False, fit=False)

In [34]:
####################################################################################################################
# A FUNCTION TO HANDLE AGGREGATION BY: COUNTRY/AREA, YEAR, FOR OPERATING PLANTS  https://docs.google.com/spreadsheets/d/1RJXc1oU569BAUUkeviR_l-Wz6HXX6mQTbxzZb-OfSPc/edit?gid=1953853820#gid=1953853820
# This updates the sheet called: "Power capacity by region and year, grouped by status"
####################################################################################################################

def cap_by_country_and_year(tech, year_min=2000, year_max=2025, dataframe=None):
    """
    Calculates operating power capacity by country/area and Start year for a specific technology type.
    Handles bi-national hydropower projects by splitting their capacity between relevant countries.

    Args:
        tech (str): The technology type to filter by (e.g., 'coal', 'hydropower').
        year_min (int): The minimum Start year to include (inclusive). Defaults to 2000.
        year_max (int): The maximum Start year to include (inclusive). Defaults to 2025.
        dataframe (pd.DataFrame, optional): The input GIPT DataFrame. If None,
                                           it uses the global `gipt` DataFrame.

    Returns:
        pd.DataFrame: A DataFrame with operating capacity aggregated by country/area
                      and Start year for the specified technology.
                      Columns represent years from `year_min` to `year_max`,
                      and includes 'region' and 'sub-region' columns.
    """
    if dataframe is None:
        df = gipt.copy()
    else:
        df = dataframe.copy()

    # Filter projects for 'operating' status and the specific technology type
    df_filtered_status_and_tech = df[(df.Status == 'operating') & (df.Type == tech)].copy()

    # Convert 'Start year' to numeric, coercing errors to NaN
    df_filtered_status_and_tech['Start year'] = pd.to_numeric(
        df_filtered_status_and_tech['Start year'], errors='coerce'
    )

    # Separate projects with missing 'Start year'
    missing_year_projects = df_filtered_status_and_tech[df_filtered_status_and_tech['Start year'].isna()].copy()
    # Filter out missing year projects from the main DataFrame for year-wise aggregation
    df_valid_years = df_filtered_status_and_tech[df_filtered_status_and_tech['Start year'].notna()].copy()
    df_valid_years['Start year'] = df_valid_years['Start year'].astype(int)

    # Further filter by the specified year range
    df_valid_years = df_valid_years[
        (df_valid_years['Start year'] >= year_min) &
        (df_valid_years['Start year'] <= year_max)
    ]

    all_years = list(range(year_min, year_max + 1))

    if tech == 'hydropower':
        # --- Hydropower-specific handling for bi-national projects (valid years) ---
        mononational_hydro_valid_years = df_valid_years[
            df_valid_years['Country/area 2 (hydropower only)'].isnull()
        ]
        binational_hydro_valid_years = df_valid_years[
            df_valid_years['Country/area 2 (hydropower only)'].notnull()
        ]

        binational_hydro_part1_valid_years = binational_hydro_valid_years[[
            'Country/area 2 (hydropower only)', 'Country/area 2 Capacity (MW) (hydropower only)',
            'Start year'
        ]].rename(columns={
            'Country/area 2 (hydropower only)': 'Country/area',
            'Country/area 2 Capacity (MW) (hydropower only)': 'Capacity (MW)'
        })

        binational_hydro_part2_valid_years = binational_hydro_valid_years[[
            'Country/area 1 (hydropower only)', 'Country/area 1 Capacity (MW) (hydropower only)',
            'Start year'
        ]].rename(columns={
            'Country/area 1 (hydropower only)': 'Country/area',
            'Country/area 1 Capacity (MW) (hydropower only)': 'Capacity (MW)'
        })

        all_hydro_data_valid_years = pd.concat([
            mononational_hydro_valid_years[['Country/area', 'Capacity (MW)', 'Start year']],
            binational_hydro_part1_valid_years[['Country/area', 'Capacity (MW)', 'Start year']],
            binational_hydro_part2_valid_years[['Country/area', 'Capacity (MW)', 'Start year']]
        ], ignore_index=True)

        agg_df = all_hydro_data_valid_years.groupby(['Country/area', 'Start year'])['Capacity (MW)'].sum().unstack(fill_value=0.0)

        # --- Hydropower-specific handling for bi-national projects (missing years) ---
        mononational_hydro_missing_year = missing_year_projects[
            missing_year_projects['Country/area 2 (hydropower only)'].isnull()
        ].copy()
        binational_hydro_missing_year = missing_year_projects[
            missing_year_projects['Country/area 2 (hydropower only)'].notnull()
        ].copy()

        binational_hydro_part1_missing_year = binational_hydro_missing_year[[
            'Country/area 2 (hydropower only)', 'Country/area 2 Capacity (MW) (hydropower only)'
        ]].rename(columns={
            'Country/area 2 (hydropower only)': 'Country/area',
            'Country/area 2 Capacity (MW) (hydropower only)': 'Capacity (MW)'
        })

        binational_hydro_part2_missing_year = binational_hydro_missing_year[[
            'Country/area 1 (hydropower only)', 'Country/area 1 Capacity (MW) (hydropower only)'
        ]].rename(columns={
            'Country/area 1 (hydropower only)': 'Country/area',
            'Country/area 1 Capacity (MW) (hydropower only)': 'Capacity (MW)'
        })

        all_hydro_data_missing_year = pd.concat([
            mononational_hydro_missing_year[['Country/area', 'Capacity (MW)']],
            binational_hydro_part1_missing_year[['Country/area', 'Capacity (MW)']],
            binational_hydro_part2_missing_year[['Country/area', 'Capacity (MW)']]
        ], ignore_index=True)

        missing_year_agg_country = all_hydro_data_missing_year.groupby('Country/area')['Capacity (MW)'].sum()


    else:
        # --- Standard handling for all other technologies (non-hydropower) ---
        agg_df = df_valid_years.groupby(['Country/area', 'Start year'])['Capacity (MW)'].sum().unstack(fill_value=0.0)

        missing_year_agg_country = missing_year_projects.groupby('Country/area')['Capacity (MW)'].sum()

    # Ensure all countries from the original gipt are present and all years are present
    all_countries_in_gipt = np.sort(gipt['Country/area'].unique())
    final_df_result = agg_df.reindex(index=all_countries_in_gipt, columns=all_years, fill_value=0.0)

    # Create a series for missing year capacity, indexed by all countries
    missing_year_series = pd.Series(0.0, index=all_countries_in_gipt, name='Capacity (MW) no year')
    missing_year_series.update(missing_year_agg_country)

    # Add region and sub-region columns
    country_info = gipt[['Country/area', 'Region', 'Subregion']].drop_duplicates(subset="Country/area").set_index('Country/area')
    final_df_result['region'] = final_df_result.index.map(country_info['Region'].get)
    final_df_result['sub-region'] = final_df_result.index.map(country_info['Subregion'].get)

    # Add the 'Capacity (MW) no year' column
    final_df_result['Capacity (MW) no year'] = missing_year_series.loc[final_df_result.index].fillna(0.0).values

    # Reorder columns to have region and sub-region first and reset index
    final_df_result = final_df_result.reset_index().rename(columns={'index': 'Country/area'})

    final_df_result.columns.name = 'Start year'

    # Dynamically create the list of columns to return to avoid hardcoding years
    cols_to_return = ['region', 'sub-region', 'Country/area'] + all_years + ['Capacity (MW) no year']

    return final_df_result[cols_to_return]

display(cap_by_country_and_year('hydropower', year_min=2000, year_max=2025)[:50])

Start year,region,sub-region,Country/area,2000,2001,2002,2003,2004,2005,2006,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Capacity (MW) no year
0,Asia,Southern Asia,Afghanistan,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0
1,Europe,Southern Europe,Albania,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,72.0,0.0,0.0,197.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Africa,Northern Africa,Algeria,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Oceania,Polynesia,American Samoa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,Europe,Southern Europe,Andorra,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,Africa,Sub-Saharan Africa,Angola,0.0,0.0,0.0,0.0,520.0,0.0,0.0,...,2770.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,Americas,Latin America and the Caribbean,Anguilla,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,Americas,Latin America and the Caribbean,Antigua and Barbuda,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,Americas,Latin America and the Caribbean,Argentina,0.0,0.0,150.0,51.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,Asia,Western Asia,Armenia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [35]:
#####################################################################
# RUN THE FUNCTION PER TAB IN TARGET SUMMARY TABLE: https://docs.google.com/spreadsheets/d/1RJXc1oU569BAUUkeviR_l-Wz6HXX6mQTbxnZb-OfSPc/edit?gid=1953853820#gid=1953853820
#####################################################################

#AUTHORIZE EDITS
gc = pygsheets.authorize(service_file='/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Work in Progress/GDRIVE_API_CREDENTIALS.json')
spreadsheet = gc.open_by_key("1RJXc1oU569BAUUkeviR_l-Wz6HXX6mQTbxnZb-OfSPc")
#Coal TAB
test = spreadsheet.worksheet('title', 'Coal')
test.set_dataframe(cap_by_country_and_year('coal'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Oil/gas TAB
test = spreadsheet.worksheet('title', 'Oil and gas')
test.set_dataframe(cap_by_country_and_year('oil/gas'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Solar TAB
test = spreadsheet.worksheet('title', 'Utility-scale solar')
test.set_dataframe(cap_by_country_and_year('utility-scale solar'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Wind TAB
test = spreadsheet.worksheet('title', 'Wind')
test.set_dataframe(cap_by_country_and_year('wind'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Nuclear TAB
test = spreadsheet.worksheet('title', 'Nuclear')
test.set_dataframe(cap_by_country_and_year('nuclear'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Hydro TAB
test = spreadsheet.worksheet('title', 'Hydropower')
test.set_dataframe(cap_by_country_and_year('hydropower'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Bio TAB
test = spreadsheet.worksheet('title', 'Bioenergy')
test.set_dataframe(cap_by_country_and_year('bioenergy'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Geo TAB
test = spreadsheet.worksheet('title', 'Geothermal')
test.set_dataframe(cap_by_country_and_year('geothermal'), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)

In [ ]:
####################################################################################################################
# A FUNCTION: PLANNED RETIREMENTS PER COUNTRY AND YEAR https://docs.google.com/spreadsheets/d/186LmHcdbZQXUS3CVJvXiwCpoMpouElidqHHD0lDrRvE/edit?gid=1953853820#gid=1953853820
# This updates the sheet called: "Power capacity by region and year, grouped by status"
####################################################################################################################

def planned_retirements_by_country_and_year(tech, year_min=2026, year_max=2050, dataframe=None):
    """
    Calculates planned retirement capacity by country/area and Retired year for a specific technology type.
    Handles bi-national hydropower projects by splitting their capacity between relevant countries.

    Args:
        tech (str): The technology type to filter by (e.g., 'coal', 'hydropower').
        year_min (int): The minimum Retired year to include (inclusive). Defaults to 2026.
        year_max (int): The maximum Retired year to include (inclusive). Defaults to 2050.
        dataframe (pd.DataFrame, optional): The input GIPT DataFrame. If None,
                                           it uses the global `gipt` DataFrame.

    Returns:
        pd.DataFrame: A DataFrame with planned retirement capacity aggregated by country/area
                      and Retired year for the specified technology.
                      Columns represent years from `year_min` to `year_max`,
                      and includes 'region' and 'sub-region' columns.
    """
    if dataframe is None:
        df = gipt.copy()
    else:
        df = dataframe.copy()

    # Filter projects for 'operating' status and a non-null 'Retired year' for the specific technology type
    df_filtered_retired = df[(df.Status == 'operating') &
                             (df['Retired year'].notnull()) &
                             (df.Type == tech)].copy()

    # Convert 'Retired year' to integer, coercing errors to NaN and then dropping rows with NaNs.
    df_filtered_retired['Retired year'] = pd.to_numeric(
        df_filtered_retired['Retired year'], errors='coerce'
    ).fillna(0).astype(int)

    # Further filter by the specified year range for 'Retired year'
    df_filtered_retired = df_filtered_retired[
        (df_filtered_retired['Retired year'] >= year_min) &
        (df_filtered_retired['Retired year'] <= year_max)
    ]

    all_years = list(range(year_min, year_max + 1))

    if tech == 'hydropower':
        # --- Hydropower-specific handling for bi-national projects --- (Split capacity)
        mononational_hydro_projects = df_filtered_retired[
            df_filtered_retired['Country/area 2 (hydropower only)'].isnull()
        ]
        binational_hydro_projects = df_filtered_retired[
            df_filtered_retired['Country/area 2 (hydropower only)'].notnull()
        ]

        binational_hydro_part1 = binational_hydro_projects[[
            'Country/area 2 (hydropower only)', 'Country/area 2 Capacity (MW) (hydropower only)',
            'Region', 'Subregion', 'Retired year'
        ]].rename(columns={
            'Country/area 2 (hydropower only)': 'Country/area',
            'Country/area 2 Capacity (MW) (hydropower only)': 'Capacity (MW)'
        })

        binational_hydro_part2 = binational_hydro_projects[[
            'Country/area 1 (hydropower only)', 'Country/area 1 Capacity (MW) (hydropower only)',
            'Region', 'Subregion', 'Retired year'
        ]].rename(columns={
            'Country/area 1 (hydropower only)': 'Country/area',
            'Country/area 1 Capacity (MW) (hydropower only)': 'Capacity (MW)'
        })

        all_hydro_data_for_aggregation = pd.concat([
            mononational_hydro_projects[['Country/area', 'Capacity (MW)', 'Retired year']],
            binational_hydro_part1[['Country/area', 'Capacity (MW)', 'Retired year']],
            binational_hydro_part2[['Country/area', 'Capacity (MW)', 'Retired year']]
        ])

        agg_df = all_hydro_data_for_aggregation.groupby(['Country/area', 'Retired year'])['Capacity (MW)'].sum().unstack(fill_value=0.0)

    else:
        # --- Standard handling for all other technologies (non-hydropower) ---
        agg_df = df_filtered_retired.groupby(['Country/area', 'Retired year'])['Capacity (MW)'].sum().unstack(fill_value=0.0)

    # Ensure all countries from the original gipt are present and all years are present
    all_countries_in_gipt = np.sort(gipt['Country/area'].unique())
    final_df_result = agg_df.reindex(index=all_countries_in_gipt, columns=all_years, fill_value=0.0)

    # Add region and sub-region columns
    country_info = gipt[['Country/area', 'Region', 'Subregion']].drop_duplicates(subset="Country/area").set_index('Country/area')
    final_df_result['region'] = final_df_result.index.map(country_info['Region'].get)
    final_df_result['sub-region'] = final_df_result.index.map(country_info['Subregion'].get)

    # Reorder columns to have region and sub-region first and reset index
    final_df_result = final_df_result[['region', 'sub-region'] + all_years]
    final_df_result = final_df_result.reset_index().rename(columns={'index': 'Country/area'})

    # Filter out rows where all year columns are zero
    final_df_result = final_df_result[final_df_result[all_years].sum(axis=1) > 0]

    final_df_result.columns.name = 'Retired year'

    # Dynamically create the list of columns to return to avoid hardcoding years
    cols_to_return = ['region', 'sub-region', 'Country/area'] + all_years

    return final_df_result[cols_to_return]

planned_retirements_by_country_and_year('nuclear', year_min=2026, year_max=2050)

Retired year,region,sub-region,Country/area,2026,2027,2028,2029,2030,2031,2032,...,2041,2042,2043,2044,2045,2046,2047,2048,2049,2050
9,Asia,Western Asia,Armenia,448.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19,Europe,Western Europe,Belgium,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
37,Americas,Northern America,Canada,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,868.0,836.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
51,Europe,Eastern Europe,Czech Republic,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
68,Europe,Northern Europe,Finland,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1062.0
101,Asia,Eastern Asia,Japan,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
142,Europe,Western Europe,Netherlands,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
167,Europe,Eastern Europe,Russia,0.0,0.0,0.0,0.0,0.0,0.0,417.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
183,Europe,Southern Europe,Slovenia,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,727.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
186,Africa,Sub-Saharan Africa,South Africa,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,970.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#####################################################################
# RUN THE FUNCTION PER TAB IN TARGET SUMMARY TABLE: https://docs.google.com/spreadsheets/d/186LmHcdbZQXUS3CVJvXiwCpoMpouElidqHHD0lDrRvE/edit?gid=1953853820#gid=1953853820
#####################################################################

#AUTHORIZE EDITS
gc = pygsheets.authorize(service_file='/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Work in Progress/GDRIVE_API_CREDENTIALS.json')
spreadsheet = gc.open_by_key("186LmHcdbZQXUS3CVJvXiwCpoMpouElidqHHD0lDrRvE")
#Coal TAB
test = spreadsheet.worksheet('title', 'Coal')
test.set_dataframe(planned_retirements_by_country_and_year('coal', year_min=2026, year_max=2050), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Oil/gas TAB
test = spreadsheet.worksheet('title', 'Oil and gas')
test.set_dataframe(planned_retirements_by_country_and_year('oil/gas', year_min=2026, year_max=2050), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)
#Nuclear TAB
test = spreadsheet.worksheet('title', 'Nuclear')
test.set_dataframe(planned_retirements_by_country_and_year('nuclear', year_min=2026, year_max=2050), 'A6', copy_index=False, copy_head=False, extend=False, fit=False)



In [ ]:
##########################################################################################
# OWNERSHIP SUMMARY TABLE: https://docs.google.com/spreadsheets/d/1h3kUE8XNR4qMK6ULeEAlVKsPgLY04mwsKLm2p456g4Q/edit?gid=1319920966#gid=1319920966&fvid=1094251619
##########################################################################################

'''
The processing steps:
1) Loop technologies and status, applying a helper function that apportions ownership:
	a) according to the [%] share in the 'Parent' field for combustion trackers and 'Owner' field for non combustion
	b) gap filling where any entity lacks a % share by equally sharing the remaining share of the total
	c) filling any blanks in Parent entity with 'unknown'
2) Makes a few labelling and formatting adjustments to the resulting dataframe
3) out puts a reformatted wide table to fit summary table representation and paste this into the Google sheet
'''

# ----- SUMMARY OF CAPACITY PER PARENT/OWNER
# ----- FIRST HANDLE COMBUSTION 'PARENTS' IN A BIG LOOP

# Helper function to parse owners and calculate proportional shares
def parse_parent_with_percentages(row):
    owners_raw = str(row["Parent(s)"])
    capacity = row["Capacity (MW)"]
    # Find all owners and optional percentages
    pattern = r'([^;\[]+?)(?:\s*\[\s*(\d+(?:\.\d+)?)\s*%\s*\])?(?:;|$)'
    matches = re.findall(pattern, owners_raw)
    owners = []
    total_percent = 0
    percent_info = []
    for owner, pct in matches:
        owner = owner.strip()
        if pct:
            percent = float(pct)
            total_percent += percent
            percent_info.append((owner, percent))
        else:
            owners.append(owner)
    # Normalize capacity by percentage or equally split if no percentages
    result = []
    if percent_info:
        for owner, percent in percent_info:
            share = capacity * (percent / 100)
            result.append({
                "Country/area": row["Country/area"],
                "Parent(s)": owner,
                "Capacity (MW)": share
            })
    elif owners:
        share = capacity / len(owners)
        for owner in owners:
            result.append({
                "Country/area": row["Country/area"],
                "Parent(s)": owner,
                "Capacity (MW)": share
            })
    return result

res=[]
for tech in ['coal','oil/gas']:
	for status,status_name in zip([['operating'],['construction'],['pre-construction'],['announced'],['construction','pre-construction','announced']],['operating','construction','pre-construction','announced','in-dev']):
		df_tmp=gipt[(gipt.Type==tech)&(gipt.Status.isin(status))]
		df_tmp.loc[df_tmp['Parent(s)'].isnull(),'Parent(s)']='unknown'
		# Expand rows using the helper function
		tmp_rows = []
		for _, row in df_tmp.iterrows():
		    tmp_rows.extend(parse_parent_with_percentages(row))
		df_tmp_expanded = pandas.DataFrame(tmp_rows)
		# Aggregate capacity
		df_tmp_aggregated = df_tmp_expanded.groupby(["Country/area", "Parent(s)"], as_index=False)["Capacity (MW)"].sum()
		# Add rank per country
		df_tmp_aggregated["Rank"] = df_tmp_aggregated.groupby("Country/area")["Capacity (MW)"].rank(method="dense", ascending=False).astype(int)
		# Sort
		df_tmp_aggregated.sort_values(by=["Country/area", "Rank"], inplace=True)
		# Calculate percentage of total capacity per country
		df_tmp_aggregated["Total Capacity (MW)"] = df_tmp_aggregated.groupby("Country/area")["Capacity (MW)"].transform("sum")
		df_tmp_aggregated["Percentage of Total Capacity (%)"] = (df_tmp_aggregated["Capacity (MW)"] / df_tmp_aggregated["Total Capacity (MW)"])
		# Calculate cumulative percentage of total capacity per country
		df_tmp_aggregated["Cumulative Percentage (%)"] = df_tmp_aggregated.groupby("Country/area")["Percentage of Total Capacity (%)"].cumsum()
		# Sort again for clarity
		df_tmp_aggregated.sort_values(by=["Country/area", "Rank"], inplace=True)
		# Add some indexing/categories
		df_tmp_aggregated['Technology']=tech
		df_tmp_aggregated['Status']=status_name
		# Repeat sets above but aggregating globally
		global_df_tmp_aggregated=df_tmp_aggregated.groupby(['Parent(s)', 'Technology']).agg({'Capacity (MW)': 'sum',}).reset_index().sort_values('Capacity (MW)',ascending=False)
		global_df_tmp_aggregated['Country/area']='World'
		global_df_tmp_aggregated["Total Capacity (MW)"] = global_df_tmp_aggregated.groupby("Country/area")["Capacity (MW)"].transform("sum")
		global_df_tmp_aggregated["Percentage of Total Capacity (%)"] = (global_df_tmp_aggregated["Capacity (MW)"] / global_df_tmp_aggregated["Total Capacity (MW)"])
		global_df_tmp_aggregated["Cumulative Percentage (%)"] = global_df_tmp_aggregated.groupby("Country/area")["Percentage of Total Capacity (%)"].cumsum()
		global_df_tmp_aggregated=global_df_tmp_aggregated[global_df_tmp_aggregated['Capacity (MW)']!=0.]
		global_df_tmp_aggregated['Rank']=global_df_tmp_aggregated['Capacity (MW)'].rank(method="dense", ascending=False).astype(int)
		global_df_tmp_aggregated['Status']=status_name
		# Stick togther country and global outputs per technology
		res.append(pandas.concat([global_df_tmp_aggregated,df_tmp_aggregated]))

out1=pandas.concat(res)
out1["Technology"] = out1["Technology"].str.title()

# ----- NOW HANDLE NON-COMBUSTION 'OWNERS' IN A BIG LOOP

# Helper function to parse owners and calculate proportional shares
def parse_owners_with_percentages(row):
    owners_raw = str(row["Owner(s)"])
    capacity = row["Capacity (MW)"]
    # Find all owners and optional percentages
    pattern = r'([^;\[]+?)(?:\s*\[\s*(\d+(?:\.\d+)?)\s*%\s*\])?(?:;|$)'
    matches = re.findall(pattern, owners_raw)
    owners = []
    total_percent = 0
    percent_info = []
    for owner, pct in matches:
        owner = owner.strip()
        if pct:
            percent = float(pct)
            total_percent += percent
            percent_info.append((owner, percent))
        else:
            owners.append(owner)
    # Normalize capacity by percentage or equally split if no percentages
    result = []
    if percent_info:
        for owner, percent in percent_info:
            share = capacity * (percent / 100)
            result.append({
                "Country/area": row["Country/area"],
                "Owner(s)": owner,
                "Capacity (MW)": share
            })
    elif owners:
        share = capacity / len(owners)
        for owner in owners:
            result.append({
                "Country/area": row["Country/area"],
                "Owner(s)": owner,
                "Capacity (MW)": share
            })
    return result

res2=[]
for tech in ['utility-scale solar','wind','hydropower','bioenergy','geothermal','nuclear']:
	for status,status_name in zip([['operating'],['construction'],['pre-construction'],['announced'],['construction','pre-construction','announced']],['operating','construction','pre-construction','announced','in-dev']):
		df_tmp=gipt[(gipt.Type==tech)&(gipt.Status.isin(status))]
		df_tmp.loc[df_tmp['Owner(s)'].isnull(),'Owner(s)']='unknown'
		# Expand rows using the helper function
		tmp_rows = []
		for _, row in df_tmp.iterrows():
		    tmp_rows.extend(parse_owners_with_percentages(row))
		df_tmp_expanded = pandas.DataFrame(tmp_rows)
		# Aggregate capacity
		df_tmp_aggregated = df_tmp_expanded.groupby(["Country/area", "Owner(s)"], as_index=False)["Capacity (MW)"].sum()
		# Add rank per country
		df_tmp_aggregated["Rank"] = df_tmp_aggregated.groupby("Country/area")["Capacity (MW)"].rank(method="dense", ascending=False).astype(int)
		# Sort
		df_tmp_aggregated.sort_values(by=["Country/area", "Rank"], inplace=True)
		# Calculate percentage of total capacity per country
		df_tmp_aggregated["Total Capacity (MW)"] = df_tmp_aggregated.groupby("Country/area")["Capacity (MW)"].transform("sum")
		df_tmp_aggregated["Percentage of Total Capacity (%)"] = (df_tmp_aggregated["Capacity (MW)"] / df_tmp_aggregated["Total Capacity (MW)"])
		# Calculate cumulative percentage of total capacity per country
		df_tmp_aggregated["Cumulative Percentage (%)"] = df_tmp_aggregated.groupby("Country/area")["Percentage of Total Capacity (%)"].cumsum()
		# Sort again for clarity
		df_tmp_aggregated.sort_values(by=["Country/area", "Rank"], inplace=True)
		df_tmp_aggregated['Technology']=tech
		df_tmp_aggregated['Status']=status_name
		# Repeat sets above but aggregating globally
		global_df_tmp_aggregated=df_tmp_aggregated.groupby(['Owner(s)', 'Technology']).agg({'Capacity (MW)': 'sum',}).reset_index().sort_values('Capacity (MW)',ascending=False)
		global_df_tmp_aggregated['Country/area']='World'
		global_df_tmp_aggregated["Total Capacity (MW)"] = global_df_tmp_aggregated.groupby("Country/area")["Capacity (MW)"].transform("sum")
		global_df_tmp_aggregated["Percentage of Total Capacity (%)"] = (global_df_tmp_aggregated["Capacity (MW)"] / global_df_tmp_aggregated["Total Capacity (MW)"])
		global_df_tmp_aggregated["Cumulative Percentage (%)"] = global_df_tmp_aggregated.groupby("Country/area")["Percentage of Total Capacity (%)"].cumsum()
		global_df_tmp_aggregated=global_df_tmp_aggregated[global_df_tmp_aggregated['Capacity (MW)']!=0.]
		global_df_tmp_aggregated['Rank']=global_df_tmp_aggregated['Capacity (MW)'].rank(method="dense", ascending=False).astype(int)
		global_df_tmp_aggregated['Status']=status_name
		# Stick togther country and global outputs per technology
		res2.append(pandas.concat([global_df_tmp_aggregated,df_tmp_aggregated]))

out2=pandas.concat(res2)
out2["Technology"] = out2["Technology"].str.title()

# RENAME TO ALLOW STICKING TOGETHER OF DATAFRAMES
out2.rename(columns={'Owner(s)': 'Parent(s)'}, inplace=True)

# STICK TOGETHER DATAFRAMES
owners_df=pandas.concat([out1,out2])[['Technology','Country/area','Status','Parent(s)','Capacity (MW)','Rank','Percentage of Total Capacity (%)','Cumulative Percentage (%)']]

# RENAME SOME THINGS
owners_df['Status'] = owners_df['Status'].replace({
    'operating': 'Operating',
    'pre-construction': 'Pre-construction',
    'construction':'Construction',
    'announced':'Announced',
    'in-dev':'Construction+Pre-construction+Announced'
})

#####################################################################################
##
## THIS SECTION CONVERTS owners_df INTO FORMAT THAT'S FILTERABLE IN THE PUBLIC SUMMARY TABLE
##
#####################################################################################

# -----------------------------
# copy long table
# -----------------------------

df = owners_df.copy()

# -----------------------------
# optional cleanup
# -----------------------------
text_cols = ["Country/area", "Technology", "Status", "Parent(s)"]
for c in text_cols:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

# make numeric columns numeric where possible
num_cols = [
    "Rank",
    "Capacity (MW)",
    "Percentage of Total Capacity (%)",
    "Cumulative Percentage (%)"
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# -----------------------------
# choose status order
# edit this if your exact labels differ
# -----------------------------
status_order = [
    "Operating",
    "Construction",
    "Pre-construction",
    "Announced",
    "Construction+Pre-construction+Announced"
]

df["Status"] = pd.Categorical(df["Status"], categories=status_order, ordered=True)

# -----------------------------
# sort rows so tie ordering is stable
# this matters because Rank has ties
# -----------------------------
sort_cols = [
    "Country/area",
    "Technology",
    "Status",
    "Rank",
    "Capacity (MW)",
    "Parent(s)"
]
sort_ascending = [True, True, True, True, False, True]

df = df.sort_values(sort_cols, ascending=sort_ascending).reset_index(drop=True)

# -----------------------------
# create a unique row id inside each country + technology + status block
# this is the key fix: do NOT pivot on Rank because Rank has ties
# -----------------------------
df["row_id"] = (
    df.groupby(["Country/area", "Technology", "Status"])
      .cumcount() + 1
)

# -----------------------------
# pivot long -> wide
# -----------------------------
value_cols = [
    "Parent(s)",
    "Capacity (MW)",
    "Rank",
    "Percentage of Total Capacity (%)",
    "Cumulative Percentage (%)"
]

wide = df.pivot(
    index=["Country/area", "Technology", "row_id"],
    columns="Status",
    values=value_cols
).reset_index()

# -----------------------------
# flatten multiindex columns
# -----------------------------
pretty_value_names = {
    "Parent(s)": "Parent entity",
    "Capacity (MW)": "Capacity (MW)",
    "Rank": "Rank",
    "Percentage of Total Capacity (%)": "% of total capacity",
    "Cumulative Percentage (%)": "Cumulative %"
}

flat_cols = []
for col in wide.columns:
    if isinstance(col, tuple):
        left, right = col
        if right == "" or pd.isna(right):
            flat_cols.append(left)
        else:
            flat_cols.append(f"{right} - {pretty_value_names[left]}")
    else:
        flat_cols.append(col)

wide.columns = flat_cols

# -----------------------------
# optional: put columns in a neat order
# only keep columns that actually exist
# -----------------------------
desired_cols = [
    "Country/area",
    "Technology",
    "row_id",

    "Operating - Rank",
    "Operating - Parent entity",
    "Operating - Capacity (MW)",
    "Operating - % of total capacity",
    "Operating - Cumulative %",

    "Construction - Rank",
    "Construction - Parent entity",
    "Construction - Capacity (MW)",
    "Construction - % of total capacity",
    "Construction - Cumulative %",

    "Pre-construction - Rank",
    "Pre-construction - Parent entity",
    "Pre-construction - Capacity (MW)",
    "Pre-construction - % of total capacity",
    "Pre-construction - Cumulative %",

    "Announced - Rank",
    "Announced - Parent entity",
    "Announced - Capacity (MW)",
    "Announced - % of total capacity",
    "Announced - Cumulative %",

    "Construction+Pre-construction+Announced - Rank",
    "Construction+Pre-construction+Announced - Parent entity",
    "Construction+Pre-construction+Announced - Capacity (MW)",
    "Construction+Pre-construction+Announced - % of total capacity",
    "Construction+Pre-construction+Announced - Cumulative %",
]

wide = wide[[c for c in desired_cols if c in wide.columns]]

# ---------------------------------
# custom sort: force Global to top, then all others alphabetical
# ---------------------------------
wide["Country/area"] = wide["Country/area"].astype(str).str.strip()

# optional debug check
print([x for x in wide["Country/area"].dropna().unique() if "glob" in x.lower()])

# build explicit country order
other_countries = sorted([c for c in wide["Country/area"].dropna().unique() if c != "World"])
country_order = ["World"] + other_countries

wide["Country/area"] = pd.Categorical(
    wide["Country/area"],
    categories=country_order,
    ordered=True
)

wide = (
    wide.sort_values(
        by=["Country/area", "Technology", "row_id"],
        ascending=[True, True, True]
    )
    .drop(columns=["row_id"], errors="ignore")
    .reset_index(drop=True)
)

# replace NaNs with blank cells for Excel output
wide = wide.copy()
wide["Country/area"] = wide["Country/area"].astype(object)
wide = wide.fillna("")

# turn Country/area back into plain text after sorting
wide["Country/area"] = wide["Country/area"].astype(str)



/tmp/ipykernel_14934/308710408.py:258: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(["Country/area", "Technology", "Status"])


[]


In [ ]:
#####################################################################
# RUN THE FUNCTION PER TAB IN TARGET SUMMARY TABLE: # https://docs.google.com/spreadsheets/d/1h3kUE8XNR4qMK6ULeEAlVKsPgLY04mwsKLm2p456g4Q/edit?gid=1319920966#gid=1319920966&fvid=1094251619
#####################################################################

#AUTHORIZE EDITS
gc = pygsheets.authorize(service_file='/content/drive/Shareddrives/GEM Shared Drive/Programs/Special and Multi-Program Projects/Global Integrated Power Tracker/Work in Progress/GDRIVE_API_CREDENTIALS.json')
spreadsheet = gc.open_by_key("1h3kUE8XNR4qMK6ULeEAlVKsPgLY04mwsKLm2p456g4Q")
#PAST IN
test = spreadsheet.worksheet('title', 'Capacity by owner, status, and technology')
test.set_dataframe(wide, 'A11', copy_index=False, copy_head=False, extend=True)
# then explicitly reapply percent format to the needed columns
test.apply_format('F6:F', {'numberFormat': {'type': 'PERCENT', 'pattern': '0.0%'}},fields='userEnteredFormat.numberFormat')
test.apply_format('G6:G', {'numberFormat': {'type': 'PERCENT', 'pattern': '0.0%'}},fields='userEnteredFormat.numberFormat')
test.apply_format('K6:K', {'numberFormat': {'type': 'PERCENT', 'pattern': '0.0%'}},fields='userEnteredFormat.numberFormat')
test.apply_format('L6:L', {'numberFormat': {'type': 'PERCENT', 'pattern': '0.0%'}},fields='userEnteredFormat.numberFormat')
test.apply_format('P6:P', {'numberFormat': {'type': 'PERCENT', 'pattern': '0.0%'}},fields='userEnteredFormat.numberFormat')
test.apply_format('Q6:Q', {'numberFormat': {'type': 'PERCENT', 'pattern': '0.0%'}},fields='userEnteredFormat.numberFormat')
test.apply_format('U6:U', {'numberFormat': {'type': 'PERCENT', 'pattern': '0.0%'}},fields='userEnteredFormat.numberFormat')
test.apply_format('V6:V', {'numberFormat': {'type': 'PERCENT', 'pattern': '0.0%'}},fields='userEnteredFormat.numberFormat')
test.apply_format('Z6:Z', {'numberFormat': {'type': 'PERCENT', 'pattern': '0.0%'}},fields='userEnteredFormat.numberFormat')
test.apply_format('AA6:AA', {'numberFormat': {'type': 'PERCENT', 'pattern': '0.0%'}},fields='userEnteredFormat.numberFormat')
